# SC4000: CPU GNN, baseline and ablation study

**For standard Colab CPU RAM (about 12–13 GB), using local storage only.**

Run all cells in order. The notebook trains the GNN reference and eight controlled variants
on **both XLA layout collections**, plus tabular baselines/ablations on all five collections.
It repeats required comparisons with three seeds, locks model/blend choices on development
data, audits reserved graphs, and exports full baseline and selected-method submission CSVs. Download the final results ZIP
and share that one file for analysis. No repository checkout is required.

**Read before starting:**

1. Use a fresh CPU runtime. Avoid keeping a previous notebook's models/data in memory.
2. If the data already exists, set `DATA_ROOT` below. It must contain `layout/` and `tile/`,
   or a parent containing `npz_all/npz/`. Otherwise enable the download option.
3. The default sweep can take hours. The exact duration depends on graph sizes and Colab.
   The run stops after a configurable nine-hour session budget and exports partial results.
4. Local results can resume while the same files survive. A deleted/reset Colab VM loses
   local files. Download checkpoints to preserve them; upload a checkpoint ZIP to resume
   in a fresh VM with the same code, packages, settings and source data.
5. This is an end-to-end **bounded CPU GNN study**, including full test prediction.
   It does not upload to Kaggle or claim a public-leaderboard score. All required comparisons
   and submissions take priority over optional tabular exploration. If the session budget
   is reached, the ZIP explicitly marks unfinished stages; rerunning resumes automatically.

There are two fixed references: **B0**, the HGB system baseline, and **G0**, a two-layer
width-32 GraphSAGE model based on your existing GNN architecture. G0 is not deliberately
weakened to manufacture gains. Strict sampling and shared splits make this a controlled
reimplementation, rather than an exact rerun of an older model. Improvements are measured,
not guaranteed; a baseline can remain the selected method if additions do not help.

## 1. Settings

Defaults target 5 GiB for each isolated worker, leave 2 GiB available system RAM, and use
two CPU threads. The monitor terminates a worker that exceeds its RAM/time/disk limit,
records the reason, and proceeds. Bounds reduce OOM risk but cannot prevent every sudden
allocation spike or Colab runtime termination.

Only one graph and one configuration row are decoded at a time. There is **no expanded
raw-array cache**. Compact feature tables are reused for the current collection, then
deleted; model/results files and source NPZ data are retained.

In [ ]:
from pathlib import Path
import os
import sys

DATA_ROOT = ""  # e.g. "/content/predict-ai-model-runtime/npz_all/npz"
DOWNLOAD_IF_MISSING = True
RUN_NAME = "sc4000_gnn_ablation_v2"
RESTORE_ZIP = ""  # optional: "/content/sc4000_gnn_ablation_v2_results.zip"
AUTO_DOWNLOAD_FINAL_ZIP = True
SAMPLE_SUBMISSION = ""  # autodetected; otherwise path to the competition template CSV

# Scientific settings. Changing them requires a new RUN_NAME.
SEEDS = [42, 137, 2026]
COLLECTION_ORDER = ["layout:xla:default", "layout:xla:random",
                    "layout:nlp:default", "layout:nlp:random", "tile:xla"]
LAYOUT_TRAIN_GRAPHS = 40
TILE_TRAIN_GRAPHS = 160
DEVELOPMENT_GRAPHS = 12
HOLDOUT_GRAPHS = 24
TRAIN_POOL_CONFIGS = 384
FIT_CONFIGS = 256
EVALUATION_CONFIGS = 1024
GNN_TRAIN_GRAPHS = 24
GNN_TRAIN_CONFIGS = 64
GNN_MAX_NODES = 512
GNN_MAX_CONFIGURABLE_NODES = 384
GNN_MAX_EDGES = 4096
GNN_EPOCHS = 20
GNN_PATIENCE = 5

# Resource settings. These may change when resuming the same run.
SESSION_HOURS = 9.0
WORKER_RAM_GIB = 5.0
SYSTEM_RESERVE_GIB = 2.0
JOB_MINUTES = 25
EXTRACTION_MINUTES = 150
GNN_JOB_MINUTES = 120
INFERENCE_MINUTES = 90

BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_DIR = BASE_DIR / RUN_NAME
SCRATCH_DIR = BASE_DIR / ("." + RUN_NAME + "_scratch")
RUNTIME_DIR = BASE_DIR / ("." + RUN_NAME + "_runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[name] = "2"
os.environ["MPLCONFIGDIR"] = str(RUNTIME_DIR / "matplotlib")

## 2. Install missing dependencies and write the embedded study code

No competitor implementation, model weights or predictions are used. Package versions,
code hashes, data fingerprints and all settings are included in the results bundle.

In [ ]:
import importlib.util
import subprocess

packages = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn",
            "joblib": "joblib", "psutil": "psutil", "threadpoolctl": "threadpoolctl",
            "matplotlib": "matplotlib", "torch": "torch>=2.6"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", *missing])

In [ ]:
# Embedded independent implementation: engine.py
SOURCE = r'''"""Independent, bounded-memory SC4000 experiments. Embedded in the Colab notebook.

Only original NPZ files and compact sampled feature matrices are read. No full
configuration array is decompressed into memory or an expanded disk cache.
"""
import argparse
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import shutil
import sys
import tempfile
import time
import traceback
import zipfile

import joblib
import numpy as np
import pandas as pd
import psutil
from scipy.stats import kendalltau, rankdata
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, SGDClassifier
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

VERSION = '2.0.0'
GIB = 1024 ** 3
COLLECTIONS = ['layout:xla:default', 'layout:xla:random',
               'layout:nlp:default', 'layout:nlp:random', 'tile:xla']
BLOCKS = ['basic', 'masked', 'position', 'opcode', 'depth', 'edge', 'topology', 'wl']


def stable_seed(value):
    return int(hashlib.sha256(str(value).encode()).hexdigest()[:8], 16)


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, indent=2, allow_nan=False, default=str))
    temp.replace(path)


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def release_memory():
    gc.collect()
    if sys.platform == 'linux':
        try:
            import ctypes
            ctypes.CDLL('libc.so.6').malloc_trim(0)
        except (OSError, AttributeError):
            pass


def guard(required=0, reserve=1.5 * GIB):
    if psutil.virtual_memory().available < required + reserve:
        raise MemoryError(f'Insufficient available RAM for {required / GIB:.2f} GiB allocation')


def header(stream):
    version = np.lib.format.read_magic(stream)
    if version == (1, 0):
        shape, order, dtype = np.lib.format.read_array_header_1_0(stream)
    elif version in ((2, 0), (3, 0)):
        shape, order, dtype = np.lib.format.read_array_header_2_0(stream)
    else:
        raise ValueError(f'Unsupported NPY version: {version}')
    if np.dtype(dtype).hasobject:
        raise ValueError('Object arrays are not allowed')
    return tuple(shape), order, np.dtype(dtype)


def metadata(path, key):
    with zipfile.ZipFile(path) as z, z.open(key + '.npy') as f:
        return header(f)


def exact_read(f, size):
    data = bytearray(size)
    view = memoryview(data)
    position = 0
    while position < size:
        count = f.readinto(view[position:])
        if not count:
            raise EOFError('Truncated NPZ member')
        position += count
    return data


def selected_rows(path, key, indices, chunk_bytes=16 * 1024 ** 2):
    """Yield one selected row at a time; discard skipped bytes in bounded chunks.

    Fortran arrays fail explicitly: silently loading their full payload is unsafe.
    """
    indices = np.asarray(indices, dtype=np.int64)
    if len(indices) and (np.any(np.diff(indices) <= 0) or indices[0] < 0):
        raise ValueError('Indices must be sorted, unique and nonnegative')
    with zipfile.ZipFile(path) as z, z.open(key + '.npy') as f:
        shape, order, dtype = header(f)
        if order:
            raise ValueError('Fortran-order configuration array: safe streaming requires C order')
        if len(indices) and indices[-1] >= shape[0]:
            raise IndexError('Configuration index out of range')
        size = math.prod(shape[1:]) * dtype.itemsize
        guard(size * 12)
        position = 0
        for index in indices:
            skip = (int(index) - position) * size
            while skip:
                chunk = f.read(min(skip, chunk_bytes))
                if not chunk:
                    raise EOFError('Truncated NPZ while skipping rows')
                skip -= len(chunk)
            raw = exact_read(f, size)
            row = np.frombuffer(raw, dtype=dtype).reshape(shape[1:])
            yield int(index), row
            del row, raw
            position = int(index) + 1


def read_small(path, key, limit=256 * 1024 ** 2):
    shape, _, dtype = metadata(path, key)
    size = math.prod(shape) * dtype.itemsize
    if size > limit:
        raise MemoryError(f'Static member {key} exceeds {limit / 1024**2:.0f} MiB guard')
    guard(size * 3)
    with np.load(path, allow_pickle=False) as data:
        return data[key]


def family(stem):
    for name in ['resnet', 'bert', 'albert', 'inception', 'efficientnet', 'transformer',
                 'retinanet', 'mask_rcnn', 'mnasnet', 'alexnet', 'magenta', 'brax',
                 'ncf', 'xception', 'electra', 'trax', 'unet', 'mlperf']:
        if name in stem.lower():
            return name
    if len(stem) >= 24 and all(c in '0123456789abcdef' for c in stem[:24]):
        return 'hashed_graph'
    return stem.split('_')[0].split('.')[0].split('-')[0]


def balanced_order(paths, seed):
    rng = np.random.default_rng(seed)
    groups = {}
    for path in sorted(paths):
        groups.setdefault(family(path.stem), []).append(path)
    for values in groups.values():
        rng.shuffle(values)
    names = sorted(groups)
    rng.shuffle(names)
    result = []
    while any(groups.values()):
        for name in names:
            if groups[name]:
                result.append(groups[name].pop())
    return result


def make_manifest(root, collection, cfg):
    folder = Path(root).joinpath(*collection.split(':'))
    train = sorted((folder / 'train').glob('*.npz'))
    valid = sorted((folder / 'valid').glob('*.npz'))
    if len(train) < 5 or not valid:
        raise ValueError(f'{collection}: need >=5 training graphs and >=1 official validation graph')
    if {p.stem for p in train} & {p.stem for p in valid}:
        raise ValueError('Graph names overlap between train and official validation')
    tile = collection.startswith('tile')
    order = balanced_order(train, cfg['split_seed'])
    ndev = min(cfg['dev_graphs'], max(2, len(order) // 5))
    dev = order[:ndev]
    fit = order[ndev:ndev + (cfg['tile_train_graphs'] if tile else cfg['layout_train_graphs'])]
    holdout = balanced_order(valid, cfg['split_seed'] + 1)[:cfg['holdout_graphs']]
    records = []
    for split, paths in [('train', fit), ('dev', dev), ('holdout', holdout)]:
        for path in sorted(paths):
            # ZIP CRCs bind resume to the source payload, independent of mount path/mtime.
            with zipfile.ZipFile(path) as z:
                source = [(i.filename, i.CRC, i.file_size) for i in z.infolist()]
            key = 'config_feat' if tile else 'node_config_feat'
            shape, order, dtype = metadata(path, key)
            n = shape[0]
            cap = cfg['train_pool_configs'] if split == 'train' else cfg['eval_configs']
            ids = np.sort(np.random.default_rng(stable_seed(f'{cfg["split_seed"]}/{collection}/{path.stem}'))
                          .choice(n, min(n, cap), replace=False))
            records.append(dict(split=split, graph=path.stem, family=family(path.stem),
                                relative_path=str(path.relative_to(root)), count=n,
                                indices=ids.tolist(), source=digest(source),
                                fortran=bool(order), row_bytes=math.prod(shape[1:]) * dtype.itemsize))
    return dict(collection=collection, records=records,
                split_note='dev graphs from official train; holdout from official valid; no test labels')


def stats(values):
    a = np.asarray(values, dtype=np.float64)
    if not a.size:
        return np.zeros(4, dtype=np.float32)
    return np.array([a.mean(), a.std(), a.min(), a.max()], dtype=np.float32)


def hash_counts(values, bins):
    return np.bincount(np.asarray(values, dtype=np.uint64).ravel().astype(np.int64) % bins,
                       minlength=bins).astype(np.float32)


def graph_context(path, cfg):
    op = read_small(path, 'node_opcode').astype(np.int64)
    feat = read_small(path, 'node_feat').astype(np.float32)
    edge = read_small(path, 'edge_index').astype(np.int64)
    n = len(op)
    if not n or edge.ndim != 2 or edge.shape[1] != 2 or feat.shape[0] != n:
        raise ValueError('Invalid graph arrays')
    if edge.size and (edge.min() < 0 or edge.max() >= n):
        raise ValueError('Edge node index out of range')
    # Official NPZ convention: [consumer, producer], opposite to common edge lists.
    consumer, producer = edge[:, 0], edge[:, 1]
    indegree = np.bincount(consumer, minlength=n)
    outdegree = np.bincount(producer, minlength=n)
    # Kahn traversal using sorted CSR adjacency, avoiding Python lists per edge.
    order = np.argsort(producer, kind='stable')
    targets = consumer[order]
    offsets = np.concatenate([[0], np.cumsum(outdegree)])
    left = indegree.copy()
    queue = np.empty(n, dtype=np.int64)
    roots = np.flatnonzero(left == 0)
    queue[:len(roots)] = roots
    tail, head = len(roots), 0
    depth = np.zeros(n, dtype=np.int64)
    while head < tail:
        u = queue[head]
        head += 1
        for v in targets[offsets[u]:offsets[u + 1]]:
            depth[v] = max(depth[v], depth[u] + 1)
            left[v] -= 1
            if left[v] == 0:
                queue[tail] = v
                tail += 1
    feat = np.nan_to_num(feat, nan=0, posinf=0, neginf=0)
    signed = np.sign(feat) * np.log1p(np.abs(feat))
    # Fixed width preserves compatibility across graphs; upstream schema is checked per collection.
    basic = np.concatenate([np.log1p([n, len(edge)]), signed.mean(0), signed.std(0),
                            signed.min(0), signed.max(0), hash_counts(op, 32) / n]).astype(np.float32)
    topology = np.concatenate([stats(np.log1p(indegree)), stats(np.log1p(outdegree)),
                               stats(np.log1p(depth)), [np.mean(left > 0)]]).astype(np.float32)
    labels = op.copy() % 64
    wl_parts = []
    for _ in range(2):
        incoming = np.bincount(consumer, weights=(labels[producer] + 1) * 37,
                               minlength=n).astype(np.int64)
        outgoing = np.bincount(producer, weights=(labels[consumer] + 1) * 53,
                               minlength=n).astype(np.int64)
        labels = (labels * 1009 + incoming * 9176 + outgoing * 6361) % 64
        wl_parts.append(hash_counts(labels, 64) / n)
    # This bounded hashed neighborhood signature is our implementation, not exact WL isomorphism testing.
    return dict(op=op, numeric=signed, edge=edge, depth=depth, basic=basic, topology=topology,
                wl=np.concatenate(wl_parts), nodes=n, cycle_fraction=float(np.mean(left > 0)))


def layout_context(path, context, cfg):
    ids = read_small(path, 'node_config_ids').astype(np.int64)
    n = context['nodes']
    if not len(ids) or ids.min() < 0 or ids.max() >= n or len(np.unique(ids)) != len(ids):
        raise ValueError('Invalid configurable node IDs')
    lookup = np.full(n, -1, dtype=np.int64)
    lookup[ids] = np.arange(len(ids))
    pairs = lookup[context['edge']]
    pairs = pairs[np.all(pairs >= 0, axis=1)]
    # Use a deterministic, configuration-independent edge cap.
    if len(pairs) > cfg['edge_cap']:
        chosen = np.sort(np.random.default_rng(913).choice(len(pairs), cfg['edge_cap'], replace=False))
        pairs = pairs[chosen]
    d = context['depth'][ids]
    bins = np.minimum(3, (d * 4 // max(1, int(context['depth'].max()) + 1)))
    return dict(ids=ids, opcode=context['op'][ids], depth=bins, pairs=pairs)


def layout_features(row, context, local, cfg):
    a = np.asarray(row)
    if a.shape != (len(local['ids']), 18):
        raise ValueError(f'Expected layout shape (configurable_nodes,18), got {a.shape}')
    if not np.isfinite(a).all() or np.any(a != np.floor(a)) or a.min() < -1 or a.max() > 5:
        raise ValueError('Layout values must be categorical indices -1..5')
    mask = a != -1
    zero = np.where(mask, a, 0).astype(np.float32)
    basic_config = np.concatenate([zero.mean(0), zero.std(0), zero.min(0), zero.max(0),
                                   [1 - mask.mean(), np.log1p(len(a))]])
    count = mask.sum(0)
    means = zero.sum(0) / np.maximum(1, count)
    variance = (zero ** 2).sum(0) / np.maximum(1, count) - means ** 2
    minimum = np.where(mask, a, 6).min(0)
    maximum = np.where(mask, a, -1).max(0)
    minimum[count == 0], maximum[count == 0] = 0, 0
    masked = np.concatenate([means, np.sqrt(np.maximum(0, variance)), minimum, maximum,
                             count / len(a)]).astype(np.float32)
    position = np.zeros((18, 7), dtype=np.float32)
    opcode_hist = np.zeros(cfg['opcode_bins'], dtype=np.float32)
    depth_hist = np.zeros(cfg['depth_bins'], dtype=np.float32)
    # Include padding as its own category; never treat axis ID as a physical magnitude here.
    categories = a.astype(np.int64) + 1
    for p in range(18):
        token = categories[:, p]
        position[p] = np.bincount(token, minlength=7) / len(a)
        opcode_hist += hash_counts(local['opcode'] * 1009 + p * 37 + token * 9176, cfg['opcode_bins'])
        depth_hist += hash_counts(local['depth'] * 1009 + p * 37 + token * 9176, cfg['depth_bins'])
    denom = max(1, len(a) * 18)
    pairs = local['pairs']
    edge_hist = np.zeros(cfg['edge_bins'], dtype=np.float32)
    if len(pairs):
        # Categorical co-occurrence, not an assertion of physical layout compatibility.
        for p in [0, 6, 12]:
            codes = categories[pairs[:, 0], p] * 7 + categories[pairs[:, 1], p]
            edge_hist += hash_counts(codes + p * 1009, cfg['edge_bins'])
        edge_hist /= len(pairs) * 3
    return dict(basic=np.concatenate([context['basic'], basic_config]).astype(np.float32),
                masked=masked, position=position.ravel(), opcode=opcode_hist / denom,
                depth=depth_hist / denom, edge=edge_hist,
                topology=context['topology'], wl=context['wl'])


def tile_features(row, context, cfg):
    a = np.asarray(row, dtype=np.float32).ravel()
    if len(a) != 24:
        raise ValueError(f'Expected 24 tile features, got {len(a)}')
    a = np.nan_to_num(a, nan=0, posinf=0, neginf=0)
    signed = np.sign(a) * np.log1p(np.abs(a))
    # Tile-specific transformed values and interactions; layout categories do not apply.
    interactions = np.outer(signed[:8], signed[:8])[np.triu_indices(8, 1)]
    return dict(basic=np.concatenate([context['basic'], a, stats(a)]).astype(np.float32),
                masked=signed, position=interactions.astype(np.float32),
                opcode=np.zeros(1, np.float32), depth=np.zeros(1, np.float32),
                edge=np.zeros(1, np.float32), topology=context['topology'], wl=context['wl'])


def extract_graph(root, record, collection, cfg, target):
    path = Path(root) / record['relative_path']
    context = graph_context(path, cfg)
    tile = collection.startswith('tile')
    local = None if tile else layout_context(path, context, cfg)
    gnn_graph = None
    if collection in cfg.get('gnn_collections', []):
        from study_gnn import compact_graph
        gnn_graph = compact_graph(context, local, cfg)
    key = 'config_feat' if tile else 'node_config_feat'
    runtime = read_small(path, 'config_runtime').astype(np.float64)
    if tile:
        normalizer = read_small(path, 'config_runtime_normalizers').astype(np.float64)
        if np.any(normalizer <= 0) or len(normalizer) != len(runtime):
            raise ValueError('Invalid tile runtime normalizers')
        runtime /= normalizer
    ids = np.asarray(record['indices'], dtype=np.int64)
    if len(runtime) != record['count'] or not np.isfinite(runtime[ids]).all() or np.any(runtime[ids] <= 0):
        raise ValueError('Invalid runtime labels')
    arrays, fingerprints = None, []
    layouts = None
    if gnn_graph is not None:
        size = len(ids)*len(gnn_graph['gnn_positions'])*18
        guard(size*3)
        layouts = np.empty((len(ids),len(gnn_graph['gnn_positions']),18),np.int8)
    for j, (index, row) in enumerate(selected_rows(path, key, ids)):
        features = tile_features(row, context, cfg) if tile else layout_features(row, context, local, cfg)
        if arrays is None:
            required = len(ids) * sum(v.size for v in features.values()) * 4
            if required > cfg['max_graph_features_mib'] * 1024**2:
                raise MemoryError('Compact graph feature matrix exceeds configured limit')
            guard(required * 3)
            arrays = {k: np.empty((len(ids), len(v)), np.float32) for k, v in features.items()}
        for name, values in features.items():
            if not np.isfinite(values).all():
                raise ValueError('Non-finite features')
            arrays[name][j] = values
        fingerprints.append(hashlib.blake2b(np.ascontiguousarray(row).tobytes(), digest_size=16).hexdigest())
        if layouts is not None:
            layouts[j] = row[gnn_graph['gnn_positions']].astype(np.int8)
    if arrays is None:
        raise ValueError('No configurations selected')
    if gnn_graph is not None:
        arrays.update(gnn_graph)
        arrays['gnn_layout'] = layouts
    np.savez_compressed(target, **arrays, y=runtime[ids], ids=ids,
                        fingerprints=np.array(fingerprints, dtype='U32'))
    return dict(sampled_configs=len(ids), unique_configs=len(set(fingerprints)),
                cycle_fraction=context['cycle_fraction'],
                gnn_coverage=None if gnn_graph is None else gnn_graph['gnn_coverage'].tolist())


def prepare(root, manifest, cfg, scratch, output):
    """Compact artifacts only, with checkpoints per graph and no raw-array cache."""
    scratch, output = Path(scratch), Path(output)
    scratch.mkdir(parents=True, exist_ok=True)
    failures, prepared = [], []
    for k, record in enumerate(manifest['records']):
        target = scratch / f'g{k:05d}.npz'
        status = scratch / f'g{k:05d}.json'
        signature = digest(dict(record=record, version=VERSION, features={
            key: cfg[key] for key in ['opcode_bins', 'depth_bins', 'edge_bins', 'edge_cap']}))
        try:
            if target.exists() and status.exists() and json.loads(status.read_text()).get('signature') == signature:
                details = json.loads(status.read_text())['details']
            else:
                print(f'Extract {k+1}/{len(manifest["records"])} {record["split"]} {record["graph"]}', flush=True)
                details = extract_graph(root, record, manifest['collection'], cfg, target)
                write_json(status, dict(signature=signature, details=details))
            prepared.append(dict(record, artifact=target.name, **details))
        except Exception as exc:
            target.unlink(missing_ok=True)
            failures.append(dict(graph=record['graph'], split=record['split'], error=str(exc)))
            print(f'Graph excluded for every experiment: {record["graph"]}: {exc}', flush=True)
        release_memory()
        if sum(p.stat().st_size for p in scratch.glob('*.npz')) > cfg['scratch_limit_gib'] * GIB:
            raise MemoryError('Compact scratch feature disk limit reached')
    write_json(output / 'extraction_failures.json', failures)
    write_json(output / 'coverage.json', prepared)
    write_json(scratch / 'prepared.json', prepared)
    counts = {s: sum(r['split'] == s for r in prepared) for s in ['train', 'dev', 'holdout']}
    if counts['train'] < 3 or counts['dev'] < 2 or counts['holdout'] < 1:
        raise ValueError(f'Insufficient shared coverage after extraction: {counts}')
    if len(failures) / max(1, len(manifest['records'])) > cfg['max_excluded_fraction']:
        raise ValueError('Too many excluded graphs; inspect extraction_failures.json')
    write_json(scratch / 'ready.json', dict(counts=counts, complete=True))


def experiments(tile=False):
    def trial(name, blocks, **kwargs):
        return dict(name=name, blocks=blocks, model='hgb', target='centered_log',
                    weighting='graph', dedup=False, row_fraction=1., graph_fraction=1.,
                    **kwargs) if not kwargs else dict(dict(name=name, blocks=blocks, model='hgb',
                    target='centered_log', weighting='graph', dedup=False,
                    row_fraction=1., graph_fraction=1.), **kwargs)
    base = ['basic']
    full = ['basic', 'masked', 'position', 'topology', 'wl'] if tile else [
        'basic', 'masked', 'position', 'opcode', 'depth', 'edge']
    result = [trial('B0_basic_hgb', base), trial('B1_masked_hgb', ['basic', 'masked']),
              trial('P0_position', ['basic', 'masked', 'position']), trial('P_full', full)]
    if not tile:
        result += [trial('P_opcode', ['basic', 'masked', 'position', 'opcode']),
                   trial('P_depth', ['basic', 'masked', 'position', 'depth']),
                   trial('P_opcode_depth', ['basic', 'masked', 'position', 'opcode', 'depth']),
                   trial('P_edge', ['basic', 'masked', 'position', 'edge'])]
    result += [trial('S_topology', base + ['topology']), trial('S_wl', base + ['wl']),
               trial('P_full_static', full + [b for b in ['topology', 'wl'] if b not in full])]
    if tile:
        # Avoid a duplicate trial: P_full already contains all tile feature blocks.
        result = [r for r in result if r['name'] != 'P_full_static']
    for block in full:
        if block != 'basic':
            result.append(trial('A_without_' + block, [b for b in full if b != block]))
    result += [trial('C_unweighted', full, weighting='none'),
               trial('C_family_weight', full, weighting='family'),
               trial('C_log_target', full, target='log'),
               trial('C_percentile_target', full, target='percentile'),
               trial('C_deduplicate', full, dedup=True),
               trial('C_half_configs', full, row_fraction=.5),
               trial('C_half_graphs', full, graph_fraction=.5),
               trial('M_extra_trees', full, model='extra_trees'),
               trial('M_mlp', full, model='mlp'),
               trial('M_linear_regression', full, model='ridge'),
               trial('M_linear_pairwise', full, model='pairwise')]
    return result


def feature_matrix(data, blocks):
    return np.concatenate([data[b] for b in blocks], axis=1).astype(np.float32, copy=False)


def training_records(records, exp, seed, cfg):
    records = [r for r in records if r['split'] == 'train']
    if exp.get('matched'):
        records = sorted(records,key=lambda r: stable_seed(f'gnn-train/{r["graph"]}'))[:cfg['gnn_train_graphs']]
    if exp['graph_fraction'] < 1:
        order = sorted(records, key=lambda r: stable_seed(f'{seed}/{r["graph"]}'))
        records = order[:max(2, math.ceil(len(order) * exp['graph_fraction']))]
    return records


def training_indices(n, graph, seed, cfg, matched=False, fraction=1.):
    cap = cfg['gnn_configs'] if matched else cfg['fit_configs']
    take = min(n,max(1,int(cap*fraction)))
    return np.sort(np.random.default_rng(stable_seed(f'{seed}/{graph}')).permutation(n)[:take])


def training_data(scratch, records, exp, seed, cfg):
    records = training_records(records,exp,seed,cfg)
    xs, ys, weights, groups = [], [], [], []
    family_counts = {}
    for r in records:
        family_counts[r['family']] = family_counts.get(r['family'], 0) + 1
    total_bytes = 0
    for group, record in enumerate(records):
        with np.load(Path(scratch) / record['artifact'], allow_pickle=False) as data:
            n = len(data['y'])
            selected = training_indices(n,record['graph'],seed,cfg,exp.get('matched',False),exp['row_fraction'])
            x = feature_matrix(data, exp['blocks'])[selected]
            y = data['y'][selected].astype(np.float64)
            if exp['dedup']:
                _, inverse = np.unique(data['fingerprints'][selected], return_inverse=True)
                unique_ids = np.unique(inverse)
                x = np.stack([x[np.flatnonzero(inverse == j)[0]] for j in unique_ids])
                y = np.array([np.median(y[inverse == j]) for j in unique_ids])
            if exp['target'] == 'percentile':
                y = (rankdata(y) - 1) / max(1, len(y) - 1)
            else:
                y = np.log1p(y)
                if exp['target'] == 'centered_log':
                    y -= np.median(y)
            w = np.ones(len(y), np.float64)
            if exp['weighting'] in ['graph', 'family']:
                w /= len(y)
            if exp['weighting'] == 'family':
                w /= family_counts[record['family']]
            total_bytes += x.nbytes
            if total_bytes > cfg['max_matrix_mib'] * 1024**2:
                raise MemoryError('Training matrix exceeds configured bound')
            guard(total_bytes * 5)
            xs.append(x); ys.append(y); weights.append(w)
            groups.append(np.full(len(y), group, np.int32))
    x, y, w, g = np.concatenate(xs), np.concatenate(ys), np.concatenate(weights), np.concatenate(groups)
    return x, y, w / w.mean(), g


def fit_model(x, y, weights, groups, exp, seed, cfg):
    kind = exp['model']
    scaler = None
    if kind in ['ridge', 'pairwise', 'mlp']:
        scaler = StandardScaler()
        x = scaler.fit_transform(x).astype(np.float32)
    if kind == 'hgb':
        model = HistGradientBoostingRegressor(max_iter=cfg['hgb_iterations'], learning_rate=.06,
                    max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=.1,
                    early_stopping=False, random_state=seed)
        model.fit(x, y, sample_weight=weights)
    elif kind == 'extra_trees':
        model = ExtraTreesRegressor(n_estimators=cfg['extra_trees'], max_depth=14,
                                     min_samples_leaf=8, n_jobs=cfg['cpu_threads'], random_state=seed)
        model.fit(x, y, sample_weight=weights)
    elif kind == 'ridge':
        model = Ridge(alpha=10.)
        model.fit(x, y, sample_weight=weights)
    elif kind == 'mlp':
        model = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=cfg['mlp_epochs'],
                             batch_size=256, early_stopping=False, random_state=seed)
        # sklearn versions differ in weight support. Weighted resampling is explicit and stable.
        rng = np.random.default_rng(seed)
        rows = rng.choice(len(y), len(y), replace=True, p=weights / weights.sum())
        model.fit(x[rows], y[rows])
    elif kind == 'pairwise':
        # A separate linear comparator, not the existing GNN or a novelty claim.
        rng = np.random.default_rng(seed)
        left, right, pair_weights = [], [], []
        for group in np.unique(groups):
            ids = np.flatnonzero(groups == group)
            i = rng.choice(ids, cfg['pairs_per_graph'])
            j = rng.choice(ids, cfg['pairs_per_graph'])
            use = y[i] != y[j]
            i, j = i[use], j[use]
            left.extend(i); right.extend(j)
            pair_weights.extend(np.repeat(weights[ids].mean(), len(i)))
        left, right = np.asarray(left, dtype=int), np.asarray(right, dtype=int)
        if not len(left):
            raise ValueError('No unequal-runtime training pairs')
        difference = x[left] - x[right]
        labels = y[left] > y[right]
        # Mirrored pairs guarantee both classes and force antisymmetric supervision.
        difference = np.concatenate([difference, -difference])
        labels = np.concatenate([labels, ~labels])
        pw = np.tile(pair_weights, 2).astype(float)
        model = SGDClassifier(loss='log_loss', alpha=.001, max_iter=300, tol=1e-4,
                               fit_intercept=False, random_state=seed)
        model.fit(difference, labels, sample_weight=pw / pw.mean())
    else:
        raise ValueError(kind)
    return dict(model=model, scaler=scaler, experiment=exp)


def predict(bundle, x):
    if bundle['scaler'] is not None:
        x = bundle['scaler'].transform(x)
    if bundle['experiment']['model'] == 'pairwise':
        return bundle['model'].decision_function(x)
    return bundle['model'].predict(x)


def scores(y, pred, tile=False):
    """Exact tau-b on the sampled set; never call these Kaggle/public LB scores."""
    if not np.isfinite(pred).all():
        raise ValueError('Non-finite predictions')
    tau = kendalltau(y, pred, variant='b').statistic
    tau = 0. if not np.isfinite(tau) else float(tau)
    order = np.argsort(pred, kind='stable')
    regret = float(y[order[:5]].min() / y.min() - 1)
    return dict(score=1 - regret if tile else tau, kendall_tau_b=tau,
                top5_regret=regret, distinct_predictions=int(len(np.unique(pred))))


def evaluate(bundle, scratch, records, split, collection, output, seed, exp_name):
    rows, prediction_rows = [], []
    for record in records:
        if record['split'] != split:
            continue
        with np.load(Path(scratch) / record['artifact'], allow_pickle=False) as data:
            x = feature_matrix(data, bundle['experiment']['blocks'])
            y, ids = data['y'], data['ids']
            pred = predict(bundle, x)
            metric = scores(y, pred, collection.startswith('tile'))
            rows.append(dict(collection=collection, split=split, graph=record['graph'],
                             family=record['family'], experiment=exp_name, seed=seed,
                             configurations=len(y), total_configurations=record['count'], **metric))
            prediction_rows.append(pd.DataFrame(dict(graph=record['graph'], config_id=ids,
                                                      runtime=y, prediction=pred)))
    if not rows:
        raise ValueError(f'No {split} graphs')
    pd.DataFrame(rows).to_csv(Path(output) / f'{split}_graphs.csv', index=False)
    pd.concat(prediction_rows, ignore_index=True).to_csv(Path(output) / f'{split}_predictions.csv.gz',
                                                      index=False, compression='gzip')
    return float(np.mean([r['score'] for r in rows]))


def run_trial(job):
    if job['experiment']['model'] == 'gnn':
        from study_gnn import train
        return train(job)
    output = Path(job['output']); output.mkdir(parents=True, exist_ok=True)
    cfg, exp, seed = job['cfg'], job['experiment'], job['seed']
    records = json.loads((Path(job['scratch']) / 'prepared.json').read_text())
    start = time.monotonic()
    with threadpool_limits(limits=cfg['cpu_threads']):
        x, y, w, groups = training_data(job['scratch'], records, exp, seed, cfg)
        fit_start = time.monotonic()
        bundle = fit_model(x, y, w, groups, exp, seed, cfg)
        fit_seconds = time.monotonic() - fit_start
        train_rows, feature_count = x.shape
        del x, y, w, groups
        release_memory()
        dev = evaluate(bundle, job['scratch'], records, 'dev', job['collection'], output, seed, exp['name'])
        joblib.dump(bundle, output / 'model.joblib', compress=3)
    write_json(output / 'result.json', dict(status='ok', experiment=exp['name'], seed=seed,
               collection=job['collection'], dev_score=dev, train_rows=train_rows,
               feature_count=feature_count, fit_seconds=fit_seconds,
               total_seconds=time.monotonic()-start, version=VERSION))


def evaluate_holdout(job):
    if job['experiment']['model'] == 'gnn':
        from study_gnn import holdout
        return holdout(job)
    bundle = joblib.load(Path(job['output']) / 'model.joblib')
    records = json.loads((Path(job['scratch']) / 'prepared.json').read_text())
    with threadpool_limits(limits=job['cfg']['cpu_threads']):
        score = evaluate(bundle, job['scratch'], records, 'holdout', job['collection'],
                         job['output'], job['seed'], job['experiment']['name'])
    write_json(Path(job['output']) / 'holdout_result.json', dict(status='ok', score=score))


def worker(job_path):
    job = json.loads(Path(job_path).read_text())
    try:
        if job['action'] == 'prepare':
            prepare(job['root'], job['manifest'], job['cfg'], job['scratch'], job['output'])
        elif job['action'] == 'trial':
            run_trial(job)
        elif job['action'] == 'holdout':
            evaluate_holdout(job)
        elif job['action'] == 'inference':
            from study_submission import infer_graph
            infer_graph(job)
        else:
            raise ValueError(job['action'])
    except Exception:
        traceback.print_exc()
        return 1
    return 0


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--job', required=True)
    args = parser.parse_args()
    sys.exit(worker(args.job))
'''
(RUNTIME_DIR / 'study_engine.py').write_text(SOURCE)
del SOURCE

In [ ]:
# Embedded independent implementation: gnn.py
SOURCE = r'''"""CPU GraphSAGE reference, controlled additions, and bounded graph preparation.

Retains the existing project's two-layer, width-32 GraphSAGE approach. Strict
sampling, corrected producer-to-consumer edges, masks and shared manifests mean
this is a controlled reimplementation, not an exact historical reproduction.
"""
import json
from pathlib import Path
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

from study_engine import (read_small, stable_seed, scores, write_json, release_memory,
                          training_records, training_indices)

GNN_COLLECTIONS = ['layout:xla:default', 'layout:xla:random']


def catalog():
    reference = dict(model='gnn', blocks=[], layers=2, width=32, masks=False,
                     context=False, message=True, opcode=True, numeric=True,
                     loss='pairwise', matched=True)
    def exp(name, **changes):
        return dict(reference, name=name, **changes)
    return [exp('G0_reference'), exp('G_add_masks', masks=True),
            exp('G_add_context', context=True), exp('G_full', masks=True, context=True),
            exp('G_no_messages', masks=True, context=True, message=False),
            exp('G_no_opcode_embedding', masks=True, context=True, opcode=False),
            exp('G_no_numeric', masks=True, context=True, numeric=False),
            exp('G_regression', masks=True, context=True, loss='regression'),
            exp('G_one_layer', masks=True, context=True, layers=1)]


def compact_graph(context, local, cfg):
    """Strict node AND edge bounds, before PyTorch allocation; no adjacency lists."""
    ids = local['ids']
    rng = np.random.default_rng(193)
    count = min(len(ids), cfg['gnn_config_nodes'], cfg['gnn_nodes'])
    positions = np.sort(rng.choice(len(ids), count, replace=False))
    seeds = ids[positions]
    keep = np.sort(seeds)
    edges = context['edge']  # [consumer, producer]
    for _ in range(2):
        if len(keep) >= cfg['gnn_nodes']:
            break
        mask = np.zeros(context['nodes'], dtype=bool)
        mask[keep] = True
        candidates = np.unique(edges[np.any(mask[edges], axis=1)].ravel())
        candidates = candidates[~mask[candidates]]
        budget = min(len(candidates), cfg['gnn_nodes']-len(keep))
        chosen = rng.choice(candidates, budget, replace=False)
        keep = np.sort(np.concatenate([keep,chosen]))
    lookup = np.full(context['nodes'], -1, np.int64)
    lookup[keep] = np.arange(len(keep))
    mapped = lookup[edges]
    mapped = mapped[np.all(mapped >= 0,axis=1)]
    before = len(mapped)
    if before > cfg['gnn_edges']:
        chosen = np.sort(rng.choice(before,cfg['gnn_edges'],replace=False))
        mapped = mapped[chosen]
    numeric = context['numeric'][keep].copy()
    numeric = (numeric-numeric.mean(0))/(numeric.std(0)+1e-6)
    config_nodes = lookup[seeds]
    assert len(keep) <= cfg['gnn_nodes'] and len(mapped) <= cfg['gnn_edges']
    return dict(gnn_numeric=numeric.astype(np.float32), gnn_opcode=context['op'][keep].clip(0,255),
                gnn_src=mapped[:,1], gnn_dst=mapped[:,0],
                gnn_config_nodes=config_nodes, gnn_positions=positions,
                gnn_coverage=np.array([context['nodes'],len(keep),len(ids),len(positions),
                                       len(edges),before,len(mapped)],dtype=np.int64))


class Ranker(nn.Module):
    def __init__(self, numeric_dim, summary_dim, exp):
        super().__init__()
        self.exp = exp
        h = exp['width']
        self.embedding = nn.Embedding(256,16)
        # Same capacity for reference/mask/context additions: unused inputs are zeros.
        self.project = nn.Linear(numeric_dim+16+18+18,h)
        self.layers = nn.ModuleList([nn.Linear(2*h,h) for _ in range(exp['layers'])])
        self.head = nn.Sequential(nn.Linear(h*4+summary_dim,h),nn.ReLU(),nn.Dropout(.05),nn.Linear(h,1))

    def forward(self, graph, layout, summary):
        numeric, opcode, src, dst, config_nodes = graph
        batch, n = len(layout),len(numeric)
        features = numeric if self.exp['numeric'] else torch.zeros_like(numeric)
        op = self.embedding(opcode)
        if not self.exp['opcode']:
            op = torch.zeros_like(op)
        values = torch.zeros((batch,n,18),dtype=torch.float32)
        masks = torch.zeros_like(values)
        valid = layout != -1
        values[:,config_nodes] = torch.where(valid,layout,0).float()/5.
        if self.exp['masks']:
            masks[:,config_nodes] = valid.float()
        h = F.relu(self.project(torch.cat([features.unsqueeze(0).expand(batch,-1,-1),
                    op.unsqueeze(0).expand(batch,-1,-1),values,masks],dim=-1)))
        degree = torch.bincount(dst,minlength=n).clamp_min(1).float()
        for layer in self.layers:
            neighbors = torch.zeros_like(h)
            if self.exp['message'] and len(src):
                neighbors.index_add_(1,dst,h[:,src])
                neighbors /= degree[None,:,None]
            h = h + F.relu(layer(torch.cat([h,neighbors],dim=-1)))
        selected = h[:,config_nodes]
        pooled = torch.cat([h.mean(1),h.max(1).values,selected.mean(1),selected.max(1).values],dim=-1)
        if not self.exp['context']:
            summary = torch.zeros_like(summary)
        return self.head(torch.cat([pooled,summary],dim=-1)).flatten()


def arrays(data):
    graph = tuple(torch.from_numpy(np.asarray(data[k]).copy()) for k in
                  ['gnn_numeric','gnn_opcode','gnn_src','gnn_dst','gnn_config_nodes'])
    # Histograms remain small, finite and label independent.
    summary = np.concatenate([data[k] for k in ['position','opcode','depth','edge']],axis=1).astype(np.float32)
    return graph, data['gnn_layout'].copy(),summary


def loss_function(pred,y,mode):
    if mode == 'regression':
        return F.smooth_l1_loss(pred,y)
    left,right = torch.triu_indices(len(pred),len(pred),offset=1)
    difference = y[left]-y[right]
    eligible = difference.abs() > .02
    if not eligible.any():
        return F.smooth_l1_loss(pred,y)
    return F.relu(.1 - difference[eligible].sign() * (pred[left[eligible]]-pred[right[eligible]])).mean()


def atomic_torch(path,obj):
    path = Path(path)
    temp = path.with_suffix('.tmp')
    torch.save(obj,temp)
    temp.replace(path)


def load_model(path):
    saved = torch.load(path,map_location='cpu',weights_only=True)
    model = Ranker(saved['numeric_dim'],saved['summary_dim'],saved['experiment'])
    model.load_state_dict(saved['state'])
    model.eval()
    return model


def predict_arrays(model,graph,layout,summary,batch):
    model.eval()
    predictions = []
    with torch.no_grad():
        for start in range(0,len(layout),batch):
            predictions.append(model(graph,torch.from_numpy(layout[start:start+batch]),
                                      torch.from_numpy(summary[start:start+batch])).numpy())
    return np.concatenate(predictions)


def evaluate(model,job,records,split,save=True,early=False):
    rows,frames = [],[]
    for record in records:
        if record['split'] != split:
            continue
        with np.load(Path(job['scratch'])/record['artifact'],allow_pickle=False) as data:
            graph,layout,summary = arrays(data)
            y,ids = data['y'],data['ids']
            if early:
                chosen = np.sort(np.random.default_rng(331).choice(len(y),min(len(y),job['cfg']['gnn_early_configs']),replace=False))
                layout,summary,y,ids = layout[chosen],summary[chosen],y[chosen],ids[chosen]
            pred = predict_arrays(model,graph,layout,summary,job['cfg']['gnn_predict_batch'])
            rows.append(dict(collection=job['collection'],split=split,graph=record['graph'],family=record['family'],
                experiment=job['experiment']['name'],seed=job['seed'],configurations=len(y),
                total_configurations=record['count'],**scores(y,pred)))
            if save:
                frames.append(pd.DataFrame(dict(graph=record['graph'],config_id=ids,runtime=y,prediction=pred)))
    if not rows:
        raise ValueError('No evaluation graphs')
    if save:
        pd.DataFrame(rows).to_csv(Path(job['output'])/f'{split}_graphs.csv',index=False)
        pd.concat(frames).to_csv(Path(job['output'])/f'{split}_predictions.csv.gz',index=False,compression='gzip')
    return float(np.mean([r['score'] for r in rows]))


def train(job):
    cfg,exp,seed = job['cfg'],job['experiment'],job['seed']
    torch.set_num_threads(cfg['cpu_threads'])
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True)
    output = Path(job['output']);output.mkdir(parents=True,exist_ok=True)
    records = json.loads((Path(job['scratch'])/'prepared.json').read_text())
    training = training_records(records,dict(matched=True,graph_fraction=1.),seed,cfg)
    with np.load(Path(job['scratch'])/training[0]['artifact']) as data:
        graph,layout,summary = arrays(data)
        numeric_dim,summary_dim = graph[0].shape[1],summary.shape[1]
    del graph,layout,summary
    model = Ranker(numeric_dim,summary_dim,exp)
    optimizer = torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=1e-4)
    start = time.monotonic()
    best,stale,first_epoch,best_epoch = -float('inf'),0,1,0
    history = []
    previous_seconds = 0.
    checkpoint = output/'latest.pt'
    if checkpoint.exists():
        state = torch.load(checkpoint,map_location='cpu',weights_only=True)
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        best,stale,first_epoch,best_epoch = state['best'],state['stale'],state['epoch']+1,state['best_epoch']
        torch.set_rng_state(state['rng'])
        history = state['history']
        previous_seconds = state.get('elapsed_seconds',0.)
    train_ids = []
    for record in training:
        ids = training_indices(len(record['indices']),record['graph'],seed,cfg,matched=True)
        train_ids.append(dict(graph=record['graph'],config_ids=np.asarray(record['indices'])[ids].tolist()))
    write_json(output/'training_manifest.json',train_ids)
    for epoch in range(first_epoch,cfg['gnn_epochs']+1):
        if stale >= cfg['gnn_patience']:
            break
        rng = np.random.default_rng(seed+epoch*1009)
        losses=[]
        for index in rng.permutation(len(training)):
            record = training[index]
            with np.load(Path(job['scratch'])/record['artifact'],allow_pickle=False) as data:
                graph,layout,summary = arrays(data)
                ids = training_indices(len(data['y']),record['graph'],seed,cfg,matched=True)
                target = np.log1p(data['y'][ids])
                target = (target-np.median(target)).astype(np.float32)
                order = rng.permutation(len(ids))
                model.train()
                for pos in range(0,len(order),cfg['gnn_batch']):
                    batch = order[pos:pos+cfg['gnn_batch']]
                    chosen=ids[batch]
                    pred=model(graph,torch.from_numpy(layout[chosen]),torch.from_numpy(summary[chosen]))
                    loss=loss_function(pred,torch.from_numpy(target[batch]),exp['loss'])
                    if not torch.isfinite(loss):
                        raise ValueError('Non-finite GNN loss')
                    optimizer.zero_grad(set_to_none=True);loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(),5.)
                    optimizer.step();losses.append(float(loss.detach()))
        dev=evaluate(model,job,records,'dev',save=False,early=True)
        if dev > best + 1e-4:
            best,stale,best_epoch=dev,0,epoch
            atomic_torch(output/'model.pt',dict(state=model.state_dict(),experiment=exp,
                         numeric_dim=numeric_dim,summary_dim=summary_dim))
        else:
            stale+=1
        history.append(dict(epoch=epoch,loss=float(np.mean(losses)),early_dev_score=dev,best_epoch=best_epoch))
        pd.DataFrame(history).to_csv(output/'history.csv',index=False)
        atomic_torch(checkpoint,dict(model=model.state_dict(),optimizer=optimizer.state_dict(),
                     best=best,stale=stale,epoch=epoch,best_epoch=best_epoch,
                     history=history,rng=torch.get_rng_state(),
                     elapsed_seconds=previous_seconds+time.monotonic()-start))
        print(f'{exp["name"]} seed {seed} epoch {epoch}: dev={dev:.4f}, best={best:.4f}',flush=True)
    fit_seconds=previous_seconds+time.monotonic()-start
    model=load_model(output/'model.pt')
    dev=evaluate(model,job,records,'dev')
    write_json(output/'result.json',dict(status='ok',experiment=exp['name'],seed=seed,
               collection=job['collection'],dev_score=dev,train_rows=sum(len(r['config_ids']) for r in train_ids),
               feature_count=numeric_dim+36,parameters=sum(p.numel() for p in model.parameters()),
               best_epoch=best_epoch,epochs=len(history),fit_seconds=fit_seconds,
               total_seconds=previous_seconds+time.monotonic()-start,version='2.0.0'))


def holdout(job):
    torch.set_num_threads(job['cfg']['cpu_threads'])
    records=json.loads((Path(job['scratch'])/'prepared.json').read_text())
    score=evaluate(load_model(Path(job['output'])/'model.pt'),job,records,'holdout')
    write_json(Path(job['output'])/'holdout_result.json',dict(status='ok',score=score))
'''
(RUNTIME_DIR / 'study_gnn.py').write_text(SOURCE)
del SOURCE

In [ ]:
# Embedded independent implementation: submission.py
SOURCE = r'''"""Full-candidate streamed prediction with frozen development-selected models."""
import json
from pathlib import Path
import time
import zipfile

import joblib
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from threadpoolctl import threadpool_limits

from study_engine import (metadata, graph_context, layout_context, layout_features, tile_features,
                          selected_rows, predict, guard, digest, write_json, COLLECTIONS)


def ranking(prediction, count, tile, template_values):
    n = len(prediction)
    if not n or not np.isfinite(prediction).all():
        raise ValueError('Empty or nonfinite predictions')
    ordered = np.argsort(prediction,kind='stable')[:min(n,count)].tolist()
    if not tile and count > n:
        raise ValueError('Layout template requests more candidates than exist')
    if count > n:
        # Preserve ONLY the supplied template suffix, never invent candidate IDs.
        # The competition template has 0;1;2;3;4 for some tile graphs with <5 candidates.
        expected = list(range(n,count))
        if not tile or template_values != list(range(count)):
            raise ValueError('Unrecognized short-tile template contract')
        ordered += expected
    validate_ranking(ordered,n,count,tile,template_values)
    return ordered


def validate_ranking(values,n,count,tile,template_values):
    if len(values)!=count or len(set(values))!=count:
        raise ValueError('Ranking length/uniqueness mismatch')
    prefix=values[:min(n,count)]
    if any(v<0 or v>=n for v in prefix):
        raise ValueError('Invalid candidate in ranking')
    if not tile and (count>n or (count==n and sorted(values)!=list(range(n)))):
        raise ValueError('Layout ranking is not a full permutation')
    if count>n and (not tile or template_values!=list(range(count)) or values[n:]!=template_values[n:]):
        raise ValueError('Short tile suffix differs from original template')


def infer_graph(job):
    cfg=job['cfg'];tile=job['collection'].startswith('tile')
    path=Path(job['path']);directory=Path(job['directory'])
    lock=job['lock']
    names=list(dict.fromkeys(['B0_basic_hgb']+list(lock['components'])))
    members=[]
    for name in names:
        for seed in cfg['seeds']:
            model_dir=directory/'trials'/name/str(seed)
            if name.startswith('G'):
                import torch
                from study_gnn import load_model
                torch.set_num_threads(cfg['cpu_threads'])
                model=load_model(model_dir/'model.pt')
            else:
                model=joblib.load(model_dir/'model.joblib')
            members.append((name,model))
    key='config_feat' if tile else 'node_config_feat'
    n=metadata(path,key)[0][0]
    guard(n*8*(len(members)+8))
    predictions=np.empty((len(members),n),np.float64)
    context=graph_context(path,cfg)
    local=None if tile else layout_context(path,context,cfg)
    neural=any(name.startswith('G') for name in names)
    bounded=None
    if neural:
        from study_gnn import compact_graph, predict_arrays
        bounded=compact_graph(context,local,cfg)
        graph=tuple(torch.from_numpy(bounded[k].copy()) for k in
                    ['gnn_numeric','gnn_opcode','gnn_src','gnn_dst','gnn_config_nodes'])
    features=[];layouts=[];indices=[]

    def flush():
        data={k:np.stack([row[k] for row in features]) for k in features[0]}
        xs={}
        if neural:
            summary=np.concatenate([data[k] for k in ['position','opcode','depth','edge']],axis=1)
            batch_layout=np.stack(layouts)
        for member,(name,model) in enumerate(members):
            if name.startswith('G'):
                pred=predict_arrays(model,graph,batch_layout,summary,cfg['gnn_predict_batch'])
            else:
                blocks=tuple(model['experiment']['blocks'])
                if blocks not in xs:xs[blocks]=np.concatenate([data[k] for k in blocks],axis=1)
                pred=predict(model,xs[blocks])
            predictions[member,indices]=pred
        features.clear();layouts.clear();indices.clear()

    with threadpool_limits(limits=cfg['cpu_threads']):
        for index,row in selected_rows(path,key,np.arange(n)):
            features.append(tile_features(row,context,cfg) if tile else layout_features(row,context,local,cfg))
            if neural:layouts.append(row[bounded['gnn_positions']].astype(np.int8))
            indices.append(index)
            if len(indices)>=cfg['inference_batch']:flush()
        if indices:flush()
    by_method={}
    for name in names:
        by_method[name]=np.mean([rankdata(predictions[i],method='average')/n
                                for i,(member,_) in enumerate(members) if member==name],axis=0)
    selected=sum(weight*by_method[name] for name,weight in lock['components'].items())
    count=len(job['template_values'])
    result=dict(ID=job['id'],count=n,components=lock['components'],
                gnn_coverage=None if bounded is None else bounded['gnn_coverage'].tolist(),
                baseline=ranking(by_method['B0_basic_hgb'],count,tile,job['template_values']),
                selected=ranking(selected,count,tile,job['template_values']))
    write_json(Path(job['output'])/'prediction.json',result)


def locate_template(root,explicit):
    if explicit:
        path=Path(explicit)
        if not path.is_file():raise FileNotFoundError(path)
        return path
    for folder in [root,root.parent,root.parent.parent,root.parent.parent.parent]:
        for name in ['sample_submission.csv','sample_submission_Eugene.csv']:
            if (folder/name).is_file():return folder/name
    raise FileNotFoundError('Set SAMPLE_SUBMISSION to the competition template CSV path')


def preflight(root,cfg):
    """Check the complete test/template contract before spending hours on training."""
    root=Path(root)
    template=pd.read_csv(locate_template(root,cfg['sample_submission']),dtype=str)
    if list(template.columns)!=['ID','TopConfigs'] or template.ID.duplicated().any() or template.isna().any().any():
        raise ValueError('Invalid sample submission template')
    if {value.rsplit(':',1)[0] for value in template.ID}!=set(cfg['collections']):
        raise ValueError('Template must cover exactly the configured collections')
    short,truncated=0,0
    for row in template.itertuples(index=False):
        collection,stem=row.ID.rsplit(':',1)
        if collection not in COLLECTIONS or '/' in stem or '\\' in stem:
            raise ValueError('Invalid template ID')
        tile=collection.startswith('tile')
        path=root.joinpath(*collection.split(':'),'test',stem+'.npz')
        shape,fortran,_=metadata(path,'config_feat' if tile else 'node_config_feat')
        if not shape[0] or fortran:
            raise ValueError(f'{row.ID}: empty or non-streamable test configurations')
        values=[int(v) for v in row.TopConfigs.split(';')]
        validate_ranking(values,shape[0],len(values),tile,values)
        short+=tile and len(values)>shape[0]
        truncated+=not tile and len(values)<shape[0]
    return dict(rows=len(template),short_tile_rows=short,layout_rows_truncated_to_template=truncated)


def export_submissions(root,output,engine_path,cfg,deadline,execute):
    template_path=locate_template(root,cfg['sample_submission'])
    template=pd.read_csv(template_path,dtype=str)
    if list(template.columns)!=['ID','TopConfigs'] or template.ID.duplicated().any() or template.isna().any().any():
        raise ValueError('Invalid sample submission template')
    actual_collections={row.rsplit(':',1)[0] for row in template.ID}
    if actual_collections!=set(cfg['collections']):
        raise ValueError('Template collections and configured collections differ')
    tests=[]
    for row in template.itertuples(index=False):
        collection,stem=row.ID.rsplit(':',1)
        if '/' in stem or '\\' in stem or collection not in COLLECTIONS:
            raise ValueError('Invalid test ID')
        path=root.joinpath(*collection.split(':'),'test',stem+'.npz')
        with zipfile.ZipFile(path) as z:
            fingerprint=[(i.filename,i.CRC,i.file_size) for i in z.infolist()]
        tests.append(dict(id=row.ID,collection=collection,relative_path=str(path.relative_to(root)),
                          template_values=[int(v) for v in row.TopConfigs.split(';')],fingerprint=fingerprint))
    # JSON round trip normalizes tuples for resume comparison.
    manifest=json.loads(json.dumps(dict(template_sha=digest(template_path.read_text()),tests=tests)))
    marker=output/'test_manifest.json'
    if marker.exists() and json.loads(marker.read_text())!=manifest:
        raise ValueError('Test inputs/template changed; use a new RUN_NAME')
    write_json(marker,manifest)
    successes=[]
    for i,record in enumerate(tests):
        directory=output/record['collection'].replace(':','_')
        lock=json.loads((directory/'selection_lock.json').read_text())
        destination=output/'test_predictions'/f'{i:05d}';destination.mkdir(parents=True,exist_ok=True)
        job=dict(action='inference',cfg=cfg,lock=lock,directory=str(directory),
                 path=str(root/record['relative_path']),output=str(destination),**record)
        print(f'Test graph {i+1}/{len(tests)}: {record["id"]}',flush=True)
        ok=execute(job,engine_path,cfg,deadline,destination/'status.json')
        if ok:successes.append(json.loads((destination/'prediction.json').read_text()))
        if ok is None:break
    if len(successes)!=len(tests):
        write_json(output/'submission_status.json',dict(status='incomplete',completed=len(successes),total=len(tests)))
        return
    for kind in ['baseline','selected']:
        rows=[]
        for record,result in zip(tests,successes):
            if result['ID']!=record['id']:raise ValueError('Prediction ID mismatch')
            validate_ranking(result[kind],result['count'],len(record['template_values']),
                             record['collection'].startswith('tile'),record['template_values'])
            rows.append(dict(ID=result['ID'],TopConfigs=';'.join(map(str,result[kind]))))
        frame=pd.DataFrame(rows)
        if not frame.ID.equals(template.ID):raise ValueError('Submission order mismatch')
        target=output/f'submission_{kind}.csv';temp=target.with_suffix('.csv.tmp')
        frame.to_csv(temp,index=False);temp.replace(target)
    write_json(output/'submission_status.json',dict(status='complete',rows=len(tests),
               template=str(template_path.name),full_candidate_inference=True,
               layout_truncated_to_template=sum(not r['collection'].startswith('tile') and len(r['template_values'])<s['count'] for r,s in zip(tests,successes)),
               short_tile_template_rows=sum(len(r['template_values'])>s['count'] for r,s in zip(tests,successes)),
               fitted_models='same frozen models used in holdout audit; no holdout retraining'))
'''
(RUNTIME_DIR / 'study_submission.py').write_text(SOURCE)
del SOURCE

In [ ]:
# Embedded independent implementation: controller.py
SOURCE = r'''"""Sequential, resumable orchestration for the standalone Colab experiment notebook."""
import importlib.metadata
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time
import zipfile

import numpy as np
import pandas as pd
import psutil

from study_engine import (BLOCKS, COLLECTIONS, GIB, VERSION, digest, experiments,
                          make_manifest, release_memory, scores, stable_seed, write_json)


def default_config():
    return dict(
        collections=COLLECTIONS.copy(), seeds=[42, 137, 2026], split_seed=729,
        layout_train_graphs=40, tile_train_graphs=160, dev_graphs=12, holdout_graphs=24,
        train_pool_configs=384, fit_configs=256, eval_configs=1024,
        opcode_bins=128, depth_bins=64, edge_bins=64, edge_cap=100000,
        hgb_iterations=180, extra_trees=80, mlp_epochs=60, pairs_per_graph=256,
        cpu_threads=2, max_graph_features_mib=96, max_matrix_mib=256,
        worker_ram_gib=5.0, system_reserve_gib=2.0, scratch_limit_gib=2.0,
        output_limit_gib=4.0, disk_reserve_gib=3.0,
        job_minutes=25, extraction_minutes=150, session_hours=9.0,
        bootstrap_repeats=1000, max_excluded_fraction=.1,
        clean_scratch=True, retry_failed=False,
        gnn_collections=['layout:xla:default', 'layout:xla:random'],
        gnn_nodes=512, gnn_config_nodes=384, gnn_edges=4096,
        gnn_train_graphs=24, gnn_configs=64, gnn_batch=16, gnn_predict_batch=32,
        gnn_epochs=20, gnn_patience=5, gnn_early_configs=128, gnn_job_minutes=120,
        inference_batch=128, inference_minutes=90, generate_submissions=True,
        sample_submission='', optional_screening=True,
    )


def scientific_config(cfg):
    resource = {'worker_ram_gib', 'system_reserve_gib', 'scratch_limit_gib', 'output_limit_gib',
                'disk_reserve_gib', 'job_minutes', 'extraction_minutes', 'session_hours',
                'clean_scratch', 'retry_failed', 'gnn_job_minutes', 'inference_minutes', 'sample_submission'}
    return {k: v for k, v in cfg.items() if k not in resource}


def validate_config(cfg):
    if not cfg['seeds'] or len(set(cfg['seeds'])) != len(cfg['seeds']):
        raise ValueError('Use distinct training seeds')
    if cfg['fit_configs'] > cfg['train_pool_configs']:
        raise ValueError('fit_configs cannot exceed train_pool_configs')
    for name in ['fit_configs', 'eval_configs', 'train_pool_configs', 'layout_train_graphs',
                 'tile_train_graphs', 'dev_graphs', 'holdout_graphs', 'hgb_iterations', 'cpu_threads',
                 'gnn_nodes','gnn_config_nodes','gnn_edges','gnn_train_graphs','gnn_configs',
                 'gnn_batch','gnn_predict_batch','gnn_epochs','gnn_patience','gnn_early_configs','inference_batch']:
        if cfg[name] < 1:
            raise ValueError(f'{name} must be positive')
    if cfg['gnn_configs'] > cfg['train_pool_configs']:
        raise ValueError('GNN configurations exceed training pool')
    if set(cfg['gnn_collections']) - {'layout:xla:default','layout:xla:random'}:
        raise ValueError('This GNN protocol targets the two XLA layout collections')
    if set(cfg['collections']) - set(COLLECTIONS):
        raise ValueError('Unknown collection')
    if cfg['worker_ram_gib'] + cfg['system_reserve_gib'] > psutil.virtual_memory().total / GIB:
        raise ValueError('Worker cap plus system reserve exceeds machine RAM')


def folder_bytes(path):
    return sum(p.stat().st_size for p in Path(path).rglob('*') if p.is_file())


def terminate(process):
    try:
        parent = psutil.Process(process.pid)
        children = parent.children(recursive=True)
        for p in children:
            p.terminate()
        parent.terminate()
        _, alive = psutil.wait_procs(children + [parent], timeout=3)
        for p in alive:
            p.kill()
    except psutil.NoSuchProcess:
        pass
    except (psutil.AccessDenied, PermissionError):
        # Still reap our own worker if host process enumeration is restricted.
        process.terminate()
    process.wait(timeout=10)


def execute(job, engine_path, cfg, deadline, status_path):
    status_path = Path(status_path)
    if status_path.exists():
        previous = json.loads(status_path.read_text())
        if previous['status'] == 'ok':
            return True
        resumable_timeout = (previous['status']=='timeout' and job.get('experiment',{}).get('model')=='gnn'
                             and (status_path.parent/'latest.pt').exists())
        if not cfg['retry_failed'] and previous['status'] not in ['session_budget', 'interrupted'] and not resumable_timeout:
            return False
    if time.monotonic() >= deadline:
        return None
    if shutil.disk_usage(status_path.parent).free < cfg['disk_reserve_gib'] * GIB:
        write_json(status_path, dict(status='disk_guard', detail='Disk reserve reached'))
        return False
    job_path = status_path.with_suffix('.job.json')
    log_path = status_path.with_suffix('.log')
    write_json(job_path, job)
    environment = os.environ.copy()
    for name in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
        environment[name] = str(cfg['cpu_threads'])
    start, peak, reason = time.monotonic(), 0, None
    maximum = cfg['extraction_minutes'] if job['action'] == 'prepare' else cfg['job_minutes']
    if job.get('experiment', {}).get('model') == 'gnn':
        maximum = cfg['gnn_job_minutes']
    if job['action'] == 'inference':
        maximum = cfg['inference_minutes']
    with log_path.open('w') as log:
        process = subprocess.Popen([sys.executable, str(engine_path), '--job', str(job_path)],
                                   stdout=log, stderr=subprocess.STDOUT, env=environment)
        try:
            last_message = start
            while process.poll() is None:
                try:
                    parent = psutil.Process(process.pid)
                    rss = sum(p.memory_info().rss for p in [parent] + parent.children(recursive=True)
                              if p.is_running())
                    peak = max(peak, rss)
                except psutil.NoSuchProcess:
                    continue
                elapsed = time.monotonic() - start
                if rss > cfg['worker_ram_gib'] * GIB:
                    reason = 'memory_guard'
                elif psutil.virtual_memory().available < cfg['system_reserve_gib'] * GIB:
                    reason = 'system_memory_guard'
                elif elapsed > maximum * 60:
                    reason = 'timeout'
                elif time.monotonic() >= deadline:
                    reason = 'session_budget'
                elif shutil.disk_usage(status_path.parent).free < cfg['disk_reserve_gib'] * GIB:
                    reason = 'disk_guard'
                if reason:
                    terminate(process)
                    break
                if time.monotonic() - last_message >= 45:
                    print(f'  still running: {elapsed/60:.1f} min, worker {rss/GIB:.2f} GiB', flush=True)
                    last_message = time.monotonic()
                time.sleep(.5)
        except BaseException:
            terminate(process)
            write_json(status_path, dict(status='interrupted', peak_rss_gib=peak/GIB))
            raise
    status = reason or ('ok' if process.returncode == 0 else 'failed')
    write_json(status_path, dict(status=status, peak_rss_gib=peak/GIB,
                                wall_seconds=time.monotonic()-start, exit_code=process.returncode))
    release_memory()
    if status != 'ok':
        print(f'  {status}: {log_path}', flush=True)
    return None if status == 'session_budget' else status == 'ok'


def read_results(output):
    rows = []
    for path in Path(output).glob('trials/*/*/result.json'):
        status = path.parent / 'status.json'
        if status.exists() and json.loads(status.read_text()).get('status') == 'ok':
            row = json.loads(path.read_text())
            row.update(json.loads(status.read_text()))
            rows.append(row)
    return pd.DataFrame(rows)


def aligned_predictions(output, names, seeds, split):
    """One graph/config ID alignment for all ranks; reject mismatches instead of intersecting."""
    result = {}
    for name in names:
        frames = []
        for seed in seeds:
            path = Path(output) / 'trials' / name / str(seed) / f'{split}_predictions.csv.gz'
            frame = pd.read_csv(path).sort_values(['graph', 'config_id']).reset_index(drop=True)
            if frames and not frame[['graph', 'config_id', 'runtime']].equals(frames[0][['graph', 'config_id', 'runtime']]):
                raise ValueError('Prediction IDs or labels differ across seeds')
            frame['rank'] = frame.groupby('graph').prediction.rank(method='average', pct=True)
            frames.append(frame)
        mean = frames[0][['graph', 'config_id', 'runtime']].copy()
        mean['prediction'] = np.mean([f['rank'].to_numpy() for f in frames], axis=0)
        if result:
            first = next(iter(result.values()))
            if not mean[['graph', 'config_id', 'runtime']].equals(first[['graph', 'config_id', 'runtime']]):
                raise ValueError('Prediction IDs differ between methods')
        result[name] = mean
    return result


def frame_score(frame, tile):
    return float(np.mean([scores(g.runtime.to_numpy(), g.prediction.to_numpy(), tile)['score']
                          for _, g in frame.groupby('graph')]))


def core_experiments(collection, cfg):
    tabular = experiments(collection.startswith('tile'))
    core = [e for e in tabular if e['name'] in ['B0_basic_hgb','B1_masked_hgb','P0_position','P_full']
            or e['name'].startswith('A_without_')]
    if collection in cfg['gnn_collections']:
        from study_gnn import catalog
        # GNN reference/additions run before secondary tabular removals.
        core = core[:1] + [dict(core[0],name='B_matched_hgb',matched=True)] + catalog() + core[1:]
    return core


def lock_selection(output, cfg, collection, names):
    path = Path(output) / 'selection_lock.json'
    if path.exists():
        return json.loads(path.read_text())
    rows = read_results(output)
    complete = [n for n in names if set(rows[rows.experiment == n].seed) == set(cfg['seeds'])]
    if set(names) != set(complete):
        raise ValueError('Every core comparison must finish all seeds before selection/holdout')
    summaries = rows[rows.experiment.isin(complete)].groupby('experiment').dev_score.agg(['mean','std'])
    summaries.sort_values('mean',ascending=False).to_csv(Path(output)/'confirmation_dev_summary.csv')
    pred = aligned_predictions(output, complete, cfg['seeds'], 'dev')
    tile = collection.startswith('tile')
    # Select actual seed ensembles, rather than assuming the best single-seed mean is best.
    tabular = max([n for n in complete if not n.startswith('G')],key=lambda n: frame_score(pred[n],tile))
    neural = [n for n in complete if n.startswith('G')]
    gnn = max(neural,key=lambda n: frame_score(pred[n],tile)) if neural else None
    candidates = [dict(components={'B0_basic_hgb':1.},dev_score=frame_score(pred['B0_basic_hgb'],tile))]
    for base in list(dict.fromkeys(['B0_basic_hgb',tabular])):
        for weight in ([0.,.25,.5,.75,1.] if gnn else [0.]):
            components = {base:1-weight}
            if gnn:
                components[gnn] = weight
            components = {n:w for n,w in components.items() if w > 0}
            blend = pred[base].copy()
            blend['prediction'] = sum(w*pred[n].prediction for n,w in components.items())
            candidates.append(dict(components=components,dev_score=frame_score(blend,tile)))
    # Stable max keeps B0 on exact ties. Holdout is never consulted here.
    selected = max(candidates,key=lambda c:c['dev_score'])
    lock = dict(winner=tabular, gnn=gnn, components=selected['components'],
                dev_score=selected['dev_score'], confirmed=complete,seeds=cfg['seeds'],
                blend_grid=candidates,source='development predictions only',
                locked_at=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()))
    write_json(path,lock)
    return lock


def paired_summary(graphs, seeds, repeats, reference='B0_basic_hgb'):
    if graphs.empty:
        return pd.DataFrame()
    baseline = graphs[graphs.experiment == reference][['graph', 'seed', 'score']].rename(columns={'score':'baseline'})
    results = []
    for name, frame in graphs.groupby('experiment'):
        paired = frame.merge(baseline, on=['graph','seed'], validate='one_to_one')
        paired['delta'] = paired.score - paired.baseline
        # Average repeats within graph; bootstrap independent graphs, not configs/seeds.
        by_graph = paired.groupby('graph').agg(delta=('delta','mean'), seeds=('seed','nunique'))
        by_graph = by_graph[by_graph.seeds == len(seeds)]
        if by_graph.empty:
            continue
        d = by_graph.delta.to_numpy()
        rng = np.random.default_rng(551)
        boot = np.array([rng.choice(d, len(d), replace=True).mean() for _ in range(repeats)])
        low, high = np.percentile(boot, [2.5,97.5])
        base_mean = float(paired.baseline.mean())
        candidate_mean = float(paired.score.mean())
        results.append(dict(experiment=name, reference=reference, baseline_score=base_mean,
                            candidate_score=candidate_mean, mean_paired_gain=float(d.mean()),
                            score_points=100*float(d.mean()),
                            relative_gain_pct=(100*float(d.mean())/base_mean if base_mean > .01 else None),
                            graphs=len(d),
                            graph_win_fraction=float(np.mean(d > 0)),
                            bootstrap_low=float(low), bootstrap_high=float(high),
                            uncertainty_note='exploratory graph bootstrap; no multiplicity correction'))
    return pd.DataFrame(results)


def summarize_collection(output, cfg, collection):
    output = Path(output)
    result = read_results(output)
    if not result.empty:
        result.to_csv(output / 'all_trials.csv', index=False)
    for split in ['dev', 'holdout']:
        paths = list(output.glob(f'trials/*/*/{split}_graphs.csv'))
        if not paths:
            continue
        frames = []
        for p in paths:
            marker = p.parent / ('holdout_status.json' if split == 'holdout' else 'status.json')
            if marker.exists() and json.loads(marker.read_text()).get('status') == 'ok':
                frames.append(pd.read_csv(p))
        if not frames:
            continue
        graphs = pd.concat(frames, ignore_index=True)
        graphs.to_csv(output / f'{split}_graph_scores.csv', index=False)
        graphs.groupby(['experiment','seed']).score.mean().reset_index().to_csv(output / f'{split}_seed_scores.csv', index=False)
        paired_summary(graphs, cfg['seeds'], cfg['bootstrap_repeats']).to_csv(output / f'{split}_paired_gains.csv', index=False)
        for reference in ['G0_reference','G_full','B_matched_hgb']:
            if reference in set(graphs.experiment):
                paired_summary(graphs,cfg['seeds'],cfg['bootstrap_repeats'],reference).to_csv(
                    output/f'{split}_paired_vs_{reference}.csv',index=False)
        graphs.groupby(['experiment','family']).score.agg(['mean','count']).to_csv(output / f'{split}_family_scores.csv')
    lock_path = output / 'selection_lock.json'
    if not lock_path.exists():
        return
    lock = json.loads(lock_path.read_text())
    names = list(dict.fromkeys(['B0_basic_hgb', lock['winner']] + list(lock['components'])))
    if not all((output / 'trials' / n / str(s) / 'holdout_result.json').exists() for n in names for s in cfg['seeds']):
        return
    pred = aligned_predictions(output, names, cfg['seeds'], 'holdout')
    b, p = pred['B0_basic_hgb'], pred[lock['winner']]
    blend = b.copy()
    blend['prediction'] = sum(w*pred[n].prediction for n,w in lock['components'].items())
    rows = []
    for name, frame in [('baseline_seed_ensemble', b), ('winner_seed_ensemble', p), ('locked_blend', blend)]:
        frame.to_csv(output / f'{name}_holdout_predictions.csv.gz', index=False, compression='gzip')
        for graph, group in frame.groupby('graph'):
            rows.append(dict(system=name, graph=graph,
                             **scores(group.runtime.to_numpy(), group.prediction.to_numpy(), collection.startswith('tile'))))
    table = pd.DataFrame(rows)
    table.to_csv(output / 'locked_systems_holdout_graphs.csv', index=False)
    table.groupby('system').score.mean().to_csv(output / 'locked_systems_holdout_scores.csv')
    paired_summary(table.rename(columns={'system':'experiment'}).assign(seed=0),[0],
                   cfg['bootstrap_repeats'],'baseline_seed_ensemble').to_csv(
                       output/'locked_systems_holdout_gains.csv',index=False)


def make_bundle(output):
    output = Path(output)
    target = output.parent / f'{output.name}_results.zip'
    temp = target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=2) as z:
        for path in output.rglob('*'):
            if path.is_file() and not path.name.endswith('.tmp'):
                z.write(path, path.relative_to(output))
    temp.replace(target)
    return target


def summary_report(output, states, cfg):
    output = Path(output)
    text = ['# SC4000 experiment run', '',
            'These are local sampled-validation scores, not Kaggle leaderboard scores.',
            'Development graphs were held out from official train. Official valid was reserved for the final audit.',
            'The selected methods/blend were locked using development scores before the holdout was evaluated.',
            'All trials use independently implemented methods. See notebook references for inspiration.', '',
            '| Collection | Status |', '|---|---|']
    for collection, state in states.items():
        text.append(f'| {collection} | {state} |')
    text += ['', 'Collection status above describes required comparisons and the holdout audit.']
    for stage in ['submission','optional']:
        path=output/f'{stage}_status.json'
        state=json.loads(path.read_text()) if path.exists() else {'status':'pending' if cfg['generate_submissions' if stage=='submission' else 'optional_screening'] else 'disabled'}
        text += ['', f'{stage.capitalize()} stage: `{json.dumps(state)}`']
    text += ['', '## Reading the results', '',
             '- `all_trials.csv`: development scores, model size, time and monitored worker peak RAM.',
             '- `holdout_paired_gains.csv`: paired gains against HGB, averaged within graph across seeds.',
             '- `holdout_paired_vs_G0_reference.csv`: additions versus the fixed GraphSAGE reference.',
             '- `holdout_paired_vs_G_full.csv`: removals versus the full GNN; compare signs carefully.',
             '- `submission_baseline.csv`, `submission_selected.csv`: full test predictions when export finishes.',
             '- `locked_systems_holdout_scores.csv`: seed ensembles and the development-selected blend.',
             '- `coverage.json`, `manifest.json`: exact graphs/configurations and data source fingerprints.',
             '- `extraction_failures.json`, worker status/log files: skipped or failed work. Do not ignore exclusions.',
             '- Models/predictions are retained for review; raw data and temporary feature matrices are excluded.',
             '', '## Limits', '',
             'Three seeds vary training configuration subsamples and model randomness, not graph splits.',
             'Layout tau-b is exact on the fixed sampled candidates. Tile top-5 regret is relative to the sampled candidates.',
             'Scores do not establish full-test/public-leaderboard performance. No Kaggle submission is made.',
             'Graph names are disjoint; related architecture families may still occur in both sets.',
             'Confidence intervals are exploratory graph bootstraps, not corrected for the many comparisons.',
             'HGB early stopping is disabled to avoid row-wise validation leakage within a graph.',
             'The MLP uses documented weighted resampling; model-family comparisons change more than one training detail.',
             'Both XLA layout collections use a bounded GraphSAGE reference, additions and removal controls.',
             'The GNN graph is sampled: inspect node/configurable-node/edge coverage before interpreting improvements.',
             'Relative gain percentages are omitted when the reference score is near zero or negative; prefer absolute tau changes.',
             'Optional tabular screening runs only after the final selection is frozen and cannot change it.',
             'Submission CSVs rank all test candidates using the audited fitted models; they are not retrained on holdout.',
             'Resource limits stop workers proactively, but cannot guarantee protection from an OS kill or Colab reset.',
             'A downloaded ZIP can restore progress; local files alone do not survive runtime deletion.', '']
    (output / 'RUN_SUMMARY.md').write_text('\n'.join(text))
    write_json(output / 'run_state.json', dict(collections=states, config=cfg))


def run_study(root, output, scratch_root, engine_path, cfg):
    validate_config(cfg)
    root, output, scratch_root = Path(root).resolve(), Path(output).resolve(), Path(scratch_root).resolve()
    if scratch_root == output or scratch_root == root or root in scratch_root.parents or output in scratch_root.parents:
        raise ValueError('Scratch must be separate from source data and saved outputs')
    if scratch_root in root.parents or scratch_root in output.parents:
        raise ValueError('Scratch cannot contain data or saved outputs')
    output.mkdir(parents=True,exist_ok=True);scratch_root.mkdir(parents=True,exist_ok=True)
    marker = scratch_root/'.sc4000_owned_scratch'
    if not marker.exists() and any(scratch_root.iterdir()):
        raise ValueError('Refusing to use a nonempty unmarked scratch directory')
    marker.touch()
    packages = ['numpy','pandas','scipy','scikit-learn','joblib','psutil']
    if cfg['gnn_collections']:
        packages.append('torch')
    versions = {n:importlib.metadata.version(n) for n in packages}
    modules = {'engine':digest(Path(engine_path).read_text()),'controller':digest(Path(__file__).read_text())}
    for name in ['gnn','submission']:
        path = Path(engine_path).parent/f'study_{name}.py'
        if path.exists():
            modules[name] = digest(path.read_text())
    signature = digest(dict(config=scientific_config(cfg),modules=modules,versions=versions))
    identity = output/'run_identity.json'
    if identity.exists() and json.loads(identity.read_text())['signature'] != signature:
        raise ValueError('Code, scientific settings or package versions changed. Use a new RUN_NAME to avoid mixing results.')
    write_json(identity,dict(signature=signature,versions=versions,config=cfg,version=VERSION))
    deadline = time.monotonic()+cfg['session_hours']*3600
    states = {c:'pending' for c in cfg['collections']}
    active = None

    def prepare_collection(collection):
        directory=output/collection.replace(':','_');directory.mkdir(exist_ok=True)
        scratch=scratch_root/directory.name
        manifest=make_manifest(root,collection,cfg)
        previous=directory/'manifest.json'
        if previous.exists() and json.loads(previous.read_text()) != manifest:
            raise ValueError(f'{collection}: source manifest changed. Use a new RUN_NAME.')
        write_json(previous,manifest)
        if not (scratch/'ready.json').exists():
            (directory/'prepare_status.json').unlink(missing_ok=True)
        ok=execute(dict(action='prepare',root=str(root),manifest=manifest,cfg=cfg,
                        scratch=str(scratch),output=str(directory)),engine_path,cfg,deadline,directory/'prepare_status.json')
        return directory,scratch,ok

    def fit(collection,directory,scratch,exp,seed,action='trial'):
        trial=directory/'trials'/exp['name']/str(seed);trial.mkdir(parents=True,exist_ok=True)
        if folder_bytes(output) > cfg['output_limit_gib']*GIB:
            return False
        job=dict(action=action,cfg=cfg,scratch=str(scratch),collection=collection,
                 experiment=exp,seed=seed,output=str(trial))
        return execute(job,engine_path,cfg,deadline,trial/('status.json' if action=='trial' else 'holdout_status.json'))

    def cleanup(scratch):
        if cfg['clean_scratch'] and marker.exists() and scratch.parent == scratch_root and scratch.exists():
            shutil.rmtree(scratch)

    try:
        # All required comparisons precede optional exploratory searches.
        for collection in cfg['collections']:
            active=collection
            directory=output/collection.replace(':','_');directory.mkdir(exist_ok=True)
            scratch=scratch_root/directory.name
            core=core_experiments(collection,cfg)
            write_json(directory/'experiment_plan.json',core)
            # Always check source identity, including when all work is already complete.
            manifest=make_manifest(root,collection,cfg)
            previous=directory/'manifest.json'
            if previous.exists() and json.loads(previous.read_text()) != manifest:
                raise ValueError(f'{collection}: source manifest changed. Use a new RUN_NAME.')
            write_json(previous,manifest)
            if (directory/'complete.json').exists():
                states[collection]='complete (resumed)';continue
            if time.monotonic() >= deadline:
                states[collection]='incomplete: session budget';continue
            print(f'\n{collection}: {len(core)} required methods × {len(cfg["seeds"])} seeds',flush=True)
            directory,scratch,ok=prepare_collection(collection)
            if not ok:
                states[collection]='incomplete: extraction/resource limit';continue
            complete=True
            for exp in core:
                for seed in cfg['seeds']:
                    print(f'{exp["name"]}, seed {seed}',flush=True)
                    ok=fit(collection,directory,scratch,exp,seed)
                    complete=complete and bool(ok)
                    if ok is None: break
                if ok is None: break
            if not complete:
                states[collection]='incomplete: required trials (inspect status/logs)'
                cleanup(scratch);continue
            lock=lock_selection(directory,cfg,collection,[e['name'] for e in core])
            print(f'Development selection locked: {lock["components"]}; auditing reserved graphs',flush=True)
            audited=True
            for exp in core:
                for seed in cfg['seeds']:
                    ok=fit(collection,directory,scratch,exp,seed,'holdout')
                    audited=audited and bool(ok)
                    if ok is None:break
                if ok is None:break
            states[collection]='complete' if audited else 'incomplete: holdout jobs'
            if audited:write_json(directory/'complete.json',dict(status='complete'))
            summarize_collection(directory,cfg,collection)
            cleanup(scratch)
            summary_report(output,states,cfg);print(f'Checkpoint: {make_bundle(output)}',flush=True)

        # Full test export is ahead of optional model/feature searches.
        if cfg['generate_submissions']:
            from study_submission import export_submissions
            if all((output/c.replace(':','_')/'complete.json').exists() for c in cfg['collections']):
                export_submissions(root,output,engine_path,cfg,deadline,execute)
            else:
                write_json(output/'submission_status.json',dict(status='pending',reason='Required core audit incomplete'))
        if cfg['optional_screening'] and all(v.startswith('complete') for v in states.values()):
            submission_status=output/'submission_status.json'
            can_screen=not cfg['generate_submissions'] or (submission_status.exists() and
                json.loads(submission_status.read_text())['status']=='complete')
            if can_screen:
                for collection in cfg['collections']:
                    if time.monotonic() >= deadline:break
                    active=collection
                    directory,scratch,ok=prepare_collection(collection)
                    if not ok:continue
                    core_names={e['name'] for e in core_experiments(collection,cfg)}
                    optional=[e for e in experiments(collection.startswith('tile')) if e['name'] not in core_names]
                    write_json(directory/'optional_plan.json',optional)
                    for exp in optional:
                        print(f'Exploratory dev-only: {exp["name"]}',flush=True)
                        if fit(collection,directory,scratch,exp,cfg['seeds'][0]) is None:break
                    cleanup(scratch)
    except KeyboardInterrupt:
        if active:states[active]='incomplete: interrupted'
        print('Interrupted. Saving completed work; rerun with the same settings to resume.',flush=True)
    except Exception as exc:
        if active:states[active]=f'incomplete: {type(exc).__name__}: {exc}'
        write_json(output/'controller_error.json',dict(type=type(exc).__name__,error=str(exc)))
        raise
    finally:
        optional_total, optional_done, optional_failed = 0, 0, 0
        for collection in cfg['collections']:
            directory=output/collection.replace(':','_')
            core_names={e['name'] for e in core_experiments(collection,cfg)}
            for exp in experiments(collection.startswith('tile')):
                if exp['name'] in core_names:continue
                optional_total+=1
                status=directory/'trials'/exp['name']/str(cfg['seeds'][0])/'status.json'
                if status.exists():
                    if json.loads(status.read_text())['status']=='ok':optional_done+=1
                    else:optional_failed+=1
            if directory.exists():
                try:summarize_collection(directory,cfg,collection)
                except Exception as exc:write_json(directory/'summary_error.json',dict(error=str(exc)))
            cleanup(scratch_root/directory.name)
        write_json(output/'optional_status.json',dict(
            status=('disabled' if not cfg['optional_screening'] else 'complete' if optional_done==optional_total else 'incomplete'),
            completed=optional_done,total=optional_total,failed_or_interrupted=optional_failed,
            interpretation='development-only exploration; does not alter the locked selection'))
        summary_report(output,states,cfg)
        bundle=make_bundle(output);release_memory()
        print(f'\nResults bundle: {bundle}\nRaw data retained; temporary feature matrices removed.',flush=True)
    return bundle
'''
(RUNTIME_DIR / 'study_controller.py').write_text(SOURCE)
del SOURCE

In [ ]:
import importlib
import json
import shutil
import zipfile
import numpy as np
import pandas as pd
import psutil

sys.path.insert(0, str(RUNTIME_DIR))
import study_engine
import study_gnn
import study_submission
import study_controller
for module in [study_engine, study_gnn, study_submission, study_controller]:
    importlib.reload(module)
from study_controller import default_config, run_study

CONFIG = default_config()
CONFIG.update(seeds=SEEDS, collections=COLLECTION_ORDER,
    layout_train_graphs=LAYOUT_TRAIN_GRAPHS, tile_train_graphs=TILE_TRAIN_GRAPHS,
    dev_graphs=DEVELOPMENT_GRAPHS, holdout_graphs=HOLDOUT_GRAPHS,
    train_pool_configs=TRAIN_POOL_CONFIGS, fit_configs=FIT_CONFIGS,
    eval_configs=EVALUATION_CONFIGS, session_hours=SESSION_HOURS,
    worker_ram_gib=WORKER_RAM_GIB, system_reserve_gib=SYSTEM_RESERVE_GIB,
    job_minutes=JOB_MINUTES, extraction_minutes=EXTRACTION_MINUTES,
    gnn_train_graphs=GNN_TRAIN_GRAPHS, gnn_configs=GNN_TRAIN_CONFIGS,
    gnn_nodes=GNN_MAX_NODES, gnn_config_nodes=GNN_MAX_CONFIGURABLE_NODES,
    gnn_edges=GNN_MAX_EDGES, gnn_epochs=GNN_EPOCHS, gnn_patience=GNN_PATIENCE,
    gnn_job_minutes=GNN_JOB_MINUTES, inference_minutes=INFERENCE_MINUTES,
    sample_submission=SAMPLE_SUBMISSION)
study_controller.validate_config(CONFIG)
print(f"RAM total: {psutil.virtual_memory().total / 1024**3:.1f} GiB")
print(f"RAM available: {psutil.virtual_memory().available / 1024**3:.1f} GiB")
print(f"Disk available: {shutil.disk_usage(BASE_DIR).free / 1024**3:.1f} GiB")
print("Output:", OUTPUT_DIR)

## 3. Locate data or download it once

If downloading, first accept the competition rules on Kaggle. Store `KAGGLE_USERNAME`
and `KAGGLE_KEY` in Colab Secrets, or use an existing authenticated Kaggle session.
The notebook does not print or include credentials in the results.

Source competition files are required disk space, not disposable model cache. They are
not deleted. For a local CPU study, do not load the complete dataset into a Python object.

In [ ]:
def locate_npz_root():
    bases = ([Path(DATA_ROOT)] if DATA_ROOT else []) + [
        BASE_DIR / "predict-ai-model-runtime", BASE_DIR / "data", BASE_DIR,
    ]
    for base in bases:
        for path in [base, base / "npz_all/npz", base / "npz"]:
            if (path / "layout").is_dir() and (path / "tile").is_dir():
                return path.resolve()
    return None

NPZ_ROOT = locate_npz_root()
if NPZ_ROOT is None and DOWNLOAD_IF_MISSING:
    from importlib.metadata import version, PackageNotFoundError
    try:
        installed = tuple(int(x) for x in version("kagglehub").split(".")[:3])
    except (PackageNotFoundError, ValueError):
        installed = (0, 0, 0)
    if installed < (0, 4, 3):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "kagglehub>=0.4.3,<2"])
    try:
        from google.colab import userdata
        for key in ["KAGGLE_USERNAME", "KAGGLE_KEY"]:
            if not os.environ.get(key):
                try:
                    value = userdata.get(key)
                    if value:
                        os.environ[key] = value
                except Exception:
                    pass
    except ImportError:
        pass
    import kagglehub
    if shutil.disk_usage(BASE_DIR).free < 15 * 1024**3:
        raise RuntimeError("Less than 15 GiB free before download. Free space or provide existing data first.")
    print("Downloading competition source files; this can take time and substantial disk space.")
    kagglehub.competition_download("predict-ai-model-runtime", output_dir=str(BASE_DIR / "predict-ai-model-runtime"))
    NPZ_ROOT = locate_npz_root()
if NPZ_ROOT is None:
    raise FileNotFoundError("Set DATA_ROOT to the extracted folder containing layout/ and tile/.")
print("NPZ source:", NPZ_ROOT)
TEMPLATE_PATH = study_submission.locate_template(NPZ_ROOT, SAMPLE_SUBMISSION)
CONFIG["sample_submission"] = str(TEMPLATE_PATH)
print("Submission template:", TEMPLATE_PATH)
print("Test/template preflight:", study_submission.preflight(NPZ_ROOT, CONFIG))

## 4. Restore an optional checkpoint and inspect the experiment plan

For a new run leave `RESTORE_ZIP` blank. To resume after losing a VM, upload a previously
downloaded results ZIP through the Colab Files panel, set its path in section 1, and run
again with the original settings. Models in the ZIP are Python joblib files: restore only
your own trusted notebook output. The original NPZ data must also be available again.

**Validation protocol:** development graphs come from the official training split;
reserved audit graphs come from the official validation split. Candidate configuration
IDs are sampled without labels and fixed in a manifest. No graph occurs in both splits
within a collection, although related architecture families can. Existing work may already
have used some official validation graphs; this is an untouched holdout for this run only,
not a claim that the project has never inspected those graphs.

In [ ]:
if RESTORE_ZIP:
    archive_path = Path(RESTORE_ZIP).resolve()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if any(OUTPUT_DIR.iterdir()):
        raise ValueError("Restore into an empty output directory to avoid overwriting results.")
    with zipfile.ZipFile(archive_path) as archive:
        if sum(item.file_size for item in archive.infolist()) > CONFIG["output_limit_gib"] * 1024**3:
            raise ValueError("Checkpoint exceeds configured output size limit.")
        for item in archive.infolist():
            destination = (OUTPUT_DIR / item.filename).resolve()
            if OUTPUT_DIR.resolve() not in destination.parents:
                raise ValueError("Unsafe checkpoint path.")
        archive.extractall(OUTPUT_DIR)
    print("Restored:", archive_path)

plan = []
for collection in CONFIG["collections"]:
    trials = study_controller.core_experiments(collection, CONFIG)
    for trial in trials:
        plan.append(dict(collection=collection, **trial))
plan_df = pd.DataFrame(plan)
display(plan_df[["collection", "name", "model", "blocks"]])
print("Required fits:", len(plan_df) * len(SEEDS))
print("Every listed method runs all three seeds; optional tabular screening follows submissions.")
print("GNN additions share architecture capacity, training IDs, graph samples and budgets.")

## What the experiments test

| Group | Experiments | Interpretation |
|---|---|---|
| System reference | `B0_basic_hgb` | Basic summaries + fixed histogram gradient boosting |
| Matched training-data control | `B_matched_hgb` | HGB trained on exactly the GNN graph/configuration IDs |
| GNN reference | `G0_reference` | Two-layer width-32 GraphSAGE, opcode embedding, numeric node features, pairwise loss |
| GNN additions | `G_add_masks`, `G_add_context`, `G_full` | Explicit padding masks; global categorical position/opcode/depth/edge summaries; both |
| GNN removals | `G_no_messages`, `G_no_opcode_embedding`, `G_no_numeric` | Remove learned message aggregation, opcode embedding, or numeric node inputs from G_full |
| GNN training/architecture controls | `G_regression`, `G_one_layer` | Replace pairwise objective with regression; reduce to one message layer |
| Padding / values | `B1_masked_hgb` | Add masks and valid-only layout summaries; tile uses signed-log values |
| Position | `P0_position` | Positional layout histograms; tile uses bounded config interactions |
| Operation and graph context | `P_opcode`, `P_depth`, `P_opcode_depth`, `P_edge`, `P_full` | Preserve which operations/graph positions receive which layouts |
| Structural references | `S_topology`, `S_wl`, `P_full_static` | Configuration-invariant graph statistics and bounded neighborhood fingerprints |
| Remove one feature block | `A_without_*` | Test whether each block helps the full feature set |
| Training controls | `C_*` | Weights, target transforms, sampled duplicate aggregation, half data |
| Model comparisons | `M_*` | Extra Trees, small MLP, linear regression and pairwise linear ranking |

The positional/category representation is independently designed. Layout edge features
are categorical co-occurrences, not a physical memory-compatibility calculation. The `wl`
block is a bounded hashed neighborhood signature, not an exact graph-isomorphism test.
`config_id` is retained only for joins and never used as a predictive feature.

Duplicate handling groups **only sampled training rows** by raw configuration equality and
uses their median runtime; evaluation IDs remain untouched. Primary experiments train
on the same candidate IDs for a given seed. Other seeds vary the training subsample and
model randomness; they are not three independent graph splits. Half-data controls are
reported separately. Model-family comparisons are not pure feature ablations.

No new ranking novelty is claimed: the project's existing GNN already uses pairwise loss.
The optional linear pairwise method is a separate comparator.

**GNN interpretation:** G0 and the additions have identical parameter counts. Disabled
mask/context inputs are zero-filled, so gains are not just from a larger prediction head.
Each graph uses deterministic, label-independent sampling with hard node, configurable-node
and edge caps. The context addition summarizes the full configurable-node set, so it may
recover information omitted by graph sampling. Coverage is recorded per graph.
`G_no_opcode_embedding` retains opcode-aware global summaries; `G_no_messages` retains
edge-aware summaries. These test the learned components, not removal of all such information.
The one-layer and regression controls deliberately change architecture/objective.

GNN training uses up to 24 graphs × 64 configurations; B0 uses up to 40 × 256. The matched
HGB control separates training-data volume from model differences. All methods use the
same fixed development and holdout candidates. Early stopping uses a fixed subset of
128 development candidates per graph; selection uses the complete sampled development set.
Checkpoints retain model, optimizer and RNG state after each epoch. No full graph/config cache
or GPU is required. The scope is both XLA layout collections; NLP and tile use tabular models.

## 5. Run the full study

This cell performs repeated core experiments, development-only selection, holdout evaluation,
full test inference, paired comparisons and checkpoint export without manual model choices.
Optional tabular screening runs last, on development data only, and cannot alter the frozen
selection after the audit. Final CSVs use the same audited fitted models, without holdout retraining.
It displays progress and worker RAM. If you interrupt it, completed jobs remain usable.
The same cell resumes successful jobs rather than retraining them. Temporary features
may need to be rebuilt after cleanup. Failures remain recorded and are not silently
replaced by a different experiment.

After each completed collection a fresh results ZIP is available beside the output
folder. You can download it from the Files sidebar while the next collection runs.
An interrupted run is labelled incomplete; partial results are never presented as a full sweep.

In [ ]:
# Save the exact implementation with the evidence, so results can be audited later.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name in ["study_engine.py", "study_gnn.py", "study_submission.py", "study_controller.py"]:
    existing = OUTPUT_DIR / name
    if existing.exists() and existing.read_bytes() != (RUNTIME_DIR / name).read_bytes():
        raise ValueError("Saved implementation differs. Use a new RUN_NAME; do not overwrite old evidence.")
    if not existing.exists():
        shutil.copy2(RUNTIME_DIR / name, existing)

STUDY_RETURNED = False
try:
    RESULTS_ZIP = run_study(NPZ_ROOT, OUTPUT_DIR, SCRATCH_DIR,
                            RUNTIME_DIR / "study_engine.py", CONFIG)
    STUDY_RETURNED = True
finally:
    # Include partial evidence even when an exception stops a collection.
    if OUTPUT_DIR.exists():
        RESULTS_ZIP = study_controller.make_bundle(OUTPUT_DIR)
    if AUTO_DOWNLOAD_FINAL_ZIP and not STUDY_RETURNED and "RESULTS_ZIP" in globals():
        try:
            from google.colab import files
            files.download(str(RESULTS_ZIP))
        except (ImportError, OSError):
            print("Download manually from the Files sidebar:", RESULTS_ZIP)

## 6. Read the results and export figures

Layout uses exact Kendall tau-b on the sampled candidates; tile uses normalized runtimes
and top-five regret relative to the sampled candidates. These are local diagnostics,
not the full official competition score. Constant predictions receive a diagnostic tau
of zero rather than disappearing from the average.

The holdout is evaluated only after model and blend choices have been saved in
`selection_lock.json`. The bootstrap resamples whole graphs after averaging paired
seed differences within each graph. With many experiments and few graphs, these are
exploratory intervals, not proof of statistical significance.

Use `holdout_paired_vs_G0_reference.csv` for adding masks/context to the GNN reference,
`holdout_paired_vs_G_full.csv` for removal controls, and `holdout_paired_gains.csv` for
comparisons against HGB. These include absolute changes, score points (100 × change),
relative percentage changes only when the reference is positive and above 0.01, and
paired graph-bootstrap intervals. A tau increase from 0.40 to 0.44 is **+0.04 tau**
(or +10% relative tau), not “4% more accurate.” Negative and uncertain gains remain visible.
Feature interactions mean the gain of the full model need not equal individual gains.

In [ ]:
import matplotlib.pyplot as plt

state = json.loads((OUTPUT_DIR / "run_state.json").read_text())
display(pd.DataFrame(list(state["collections"].items()), columns=["collection", "status"]))
for collection in CONFIG["collections"]:
    directory = OUTPUT_DIR / collection.replace(":", "_")
    summary = directory / "holdout_paired_gains.csv"
    print("\n", collection)
    if summary.exists() and summary.stat().st_size > 2:
        table = pd.read_csv(summary).sort_values("mean_paired_gain")
        display(table)
        fig, ax = plt.subplots(figsize=(9, max(3, len(table) * .35)))
        ax.errorbar(table.mean_paired_gain, np.arange(len(table)),
                    xerr=np.maximum(0, np.vstack([table.mean_paired_gain-table.bootstrap_low,
                                    table.bootstrap_high-table.mean_paired_gain])), fmt="o", capsize=3)
        ax.set_yticks(np.arange(len(table)), table.experiment)
        ax.axvline(0, color="gray", linestyle="--")
        ax.set_xlabel("Paired holdout score change versus B0 (higher is better)")
        ax.set_title(collection + " — exploratory graph-bootstrap intervals")
        fig.tight_layout()
        fig.savefig(directory / "holdout_paired_gains.png", dpi=150)
        plt.show()
        plt.close(fig)
    else:
        print("No completed holdout comparisons yet. Inspect all_trials.csv and logs.")
for collection in CONFIG["gnn_collections"]:
    directory = OUTPUT_DIR / collection.replace(":", "_")
    for reference in ["G0_reference", "G_full", "B_matched_hgb"]:
        path = directory / f"holdout_paired_vs_{reference}.csv"
        if path.exists() and path.stat().st_size > 2:
            print(collection, "compared with", reference)
            display(pd.read_csv(path))
status_path = OUTPUT_DIR / "submission_status.json"
if status_path.exists():
    print("Submission export:", json.loads(status_path.read_text()))
for path in OUTPUT_DIR.glob("submission_*.csv"):
    print("CSV:", path)
RESULTS_ZIP = study_controller.make_bundle(OUTPUT_DIR)
print("Final ZIP including charts:", RESULTS_ZIP)

## 7. Download this one file

Share **the results ZIP**, not just screenshots or the winning score. It contains per-graph
scores, all sampled predictions, repeat variability, feature/model definitions, timings,
observed worker RAM, graph manifests, failures, selected models and the exact source.
It also contains `submission_baseline.csv` and `submission_selected.csv` when full test
inference finishes. In the rare tile graphs with fewer candidates than the template's five
slots, all real candidates come first and the original template suffix is preserved and
logged. All candidates are scored, then each row is truncated to its template length.
For this dataset, two layout files have 1,001 candidates but the template requests 1,000.
CSV validation checks IDs, order, uniqueness, index bounds and full permutations whenever
the template requests every candidate.
It excludes raw competition data and temporary feature tables.

The ZIP produced in section 5 already contains all numerical evidence. This final copy
also includes the charts. If the browser blocks downloads, use the Files sidebar.

In [ ]:
print(RESULTS_ZIP)
print(f"Bundle size: {RESULTS_ZIP.stat().st_size / 1024**2:.1f} MiB")
if AUTO_DOWNLOAD_FINAL_ZIP:
    try:
        from google.colab import files
        files.download(str(RESULTS_ZIP))
    except ImportError:
        print("Outside Colab: copy the ZIP from the path above.")

## References and attribution

These sources motivated questions about representation and ranking. No competitor code,
weights or predictions are embedded here. Independent implementation does not establish
global novelty; cite these inspirations in the course report.

- [1st-place write-up](https://www.kaggle.com/competitions/predict-ai-model-runtime/writeups/viet-br-connection-1st-place-solution-for-the-goog): preprocessing and configuration comparison.
- [2nd-place write-up](https://www.kaggle.com/competitions/predict-ai-model-runtime/writeups/latenciaga-2nd-place-solution-for-the-google-fast-): ranking objectives and validation variability.
- [3rd-place author repository](https://github.com/jafluri/kaggle_tpu_graph): meaningful features and repeated training.
- [5th-place write-up](https://www.kaggle.com/competitions/predict-ai-model-runtime/discussion/456093): dimension correspondence.
- [Official data description](https://www.kaggle.com/competitions/predict-ai-model-runtime/data): array meanings, edge direction and tile runtime normalization.
- [PyTorch checkpoints](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html): model, optimizer and RNG state are retained for epoch-level recovery.
- [SciPy Kendall tau-b](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kendalltau.html): tie-aware sampled-set diagnostic.
- [HGB documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html): sample weights and early-stopping settings.
- [Colab resource limits](https://research.google.com/colaboratory/faq.html): resources and session duration are not guaranteed.
- [KaggleHub download documentation](https://github.com/Kaggle/kagglehub): direct download to a chosen directory.